**Runs on Kaggle: [kaggle.com/code/jhffmn/threadatlas-graphrag-submission](https://www.kaggle.com/code/jhffmn/threadatlas-graphrag-submission)**

This notebook is ThreadAtlas's retrieval, and the GraphRAG-Bench submission run. It runs on Kaggle with the inputs attached there; this copy is the version that made its output, [answers, evaluator output and board rows](https://www.kaggle.com/code/jhffmn/threadatlas-graphrag-submission/output).

# ThreadAtlas retrieval

Given a question and the documents to search, find the stored records that answer it and hand
them, whole and dated, to a reader model, each fact with the passage of the source it was read
from. This notebook reads the store the global layer built, and the documents' text from Step 0
by reference: the store keeps a fact's unit and offsets, Step 0 keeps the text they point into.
No model call until the reader. The same code runs on a book, a series of books, or one user's
chat history; nothing branches on the question or the corpus.

It then runs every benchmark question through that path and through the baselines, scores the
answers with each benchmark's own published evaluator, and prints the tables. One global graph
holds a corpus of mixed documents. A test filters the graph as its benchmark defines it, and the
same graph is also run with no filter.

## What comes in

- The output of the [global-layer notebook](https://www.kaggle.com/code/jhffmn/threadatlas-global-layer):
  `threadatlas.sqlite`, `threadatlas.npy`, `calls.jsonl`, `library-groups.json`. 24,071 documents,
  1,231,253 facts, 54,058 cells, 52,469 abstracts, 131,345 parents, 1,627,673 vectors.
- The public datasets [ThreadAtlas Step 0](https://www.kaggle.com/datasets/jhffmn/it494-threadatlas-step0)
  (`documents.jsonl`, every document's full text, and `units.jsonl`, each unit's start and end in it)
  and [Step 1](https://www.kaggle.com/datasets/jhffmn/it494-threadatlas-step1), for the instruments.
- The raw corpus dataset, for `longmemeval_s.json`. GraphRAG-Bench's files and its evaluator are
  downloaded from one pinned commit. Every benchmark file is checked against its SHA-256.
- For GraphRAG-Bench's Medical set, a second store: the global layer's output over the medical
  corpus alone, with the [Medical build](https://www.kaggle.com/code/jhffmn/threadatlas-medical)'s
  Step 0. Attached beside the main store, it is opened too, and the Medical questions search it.

| table | what retrieval uses it for |
|---|---|
| `document` | the title and date printed on every record; the filter is a set of `doc_id` |
| `unit` | a record's chapter or chat turn, its date, and which units sit beside it |
| `node` | the name of the entity a record is about |
| `fact`, `cell`, `abstract`, `chunk` | the four kinds of record the reader can be given; a fact's `unit_id`, `quote_start` and `quote_end`, and a chunk's `start` and `end`, point into the Step 0 text |
| `fact_in_chunk` | the chunks that hold a fact's quote, which the walk steps between |
| `instance_of` | each entity's parent, and through it the same entity in other documents |
| `vec_row` | which record each row of `threadatlas.npy` belongs to |
| `search` | the FTS5 full-text index, one row per record |


## What goes out

| file | what it holds |
|---|---|
| `contexts/<set>-<arm>.jsonl` | one line per question: the context, its tokens, its seconds, its documents, and how many entries were parents |
| `answers/<set>-<arm>.jsonl` | one line per question: the response and, for LongMemEval, the judge's verdict |
| `scores/<set>-<arm>-generation.json`, `scores/<set>-<arm>-retrieval.json` | the GraphRAG-Bench evaluator's own output |
| `questions.jsonl`, `calls.jsonl` | the log of every retrieval, and every reader and judge call with its cost |
| `hand-check.jsonl` | 30 judged LongMemEval answers, drawn at random, to read by hand |
| `traced.txt` | the five questions checked by eye, with their contexts |
| `results.json` | every number in the printed tables |

## Records and vectors

A **record** is one thing the reader can be shown: a **fact** (a subject, predicate and object
with the verbatim quote it was read from), a **cell** (what one unit says about one entity), an
**abstract** (a document's or an entity's summary), or a **chunk** (256 tokens of the document's own
text, the stretch plain RAG retrieves, so a search can reach a passage no fact was read from). A fact
or a chunk has one vector. A cell or an abstract has
one vector per sentence. A **parent** summary has vectors too, but a parent is never shown to the
reader: finding one only leads to its instances. A record's similarity to the question is the
similarity of its **best** sentence, and a record is always packed whole, never cut.

A fact goes down to its source: fact, then its unit, then the raw text of that unit in Step 0.
The reader gets the fact and the **passage** it was read from. For a chat the passage is the whole
turn, since a turn is one unit. For a book or paper, whose units are chapters, it is the paragraph
around the quote, inside the unit (a blank line ends a paragraph). A passage longer than 500
characters is cut to the quote and about 250 characters on either side, at whole words. Facts read
from one passage are packed together with it. When the passage is shown, the fact's own line is
left out.

## The algorithm

The design is `docs/retrieval.md`: search decides what to show, and facts with their source decide
what counts as evidence. Numbers in brackets point to the references at the end of the notebook.

```
INPUT      question   text
           filter     a set of documents (a history, a book), or none
           asked on   the date the question is asked, when known
SETTING    hops     0, 1 or 2: how far the pool walks from the hits
           budget   tokens of context, one of BUDGETS = 2,000 and 12,000
           filter   on or off
           search   fused, or the vector search alone
           parents  on or off
CONSTANTS  K_ENTRY = 80   HITS = 40   RRF_K = 60   PARENT_INSTANCES = 10
           K_HOP = 10   FRONTIER = 20   DISCOUNT = 0.8   MAX_PASSAGE = 500 characters

1. SCORE EVERY VECTOR                                                         [4] [3]
   q = embed(question)                         384 numbers, length 1
   for each row i of the vector array:
       sim[i] = q . v[i]                       cosine, one matrix product
   score(record) = the largest sim[i] among the record's rows
       a fact has one row; a cell or an abstract has one row per sentence

2. VECTOR HITS
   drop every row whose document is outside the filter
   keep a parent's rows only when parents are on and one of its instances is inside the filter
   V = the hits records with the highest score           ties broken by record id
   with a floor: V keeps the records above its mean plus one standard deviation, at least 5

3. KEYWORD HITS                                                               [2]
   match = the question's words joined by OR
   W = the hits records inside the filter that BM25 ranks best for match
   the vector search alone leaves W empty

4. FUSE THE TWO LISTS                                                         [1]
   for each record r in V or W:
       rrf(r) = sum, over the lists that hold r, of 1 / (RRF_K + rank of r in that list)
   entries = the records ordered by rrf, highest first

5. THE POOL: THE HITS, AND WHAT A PARENT STANDS FOR
   pool = empty
   for each entry e that is not a parent:
       final(e) = rrf(e), exact ties broken by score(e)
   for each entry p that is a parent:                  a parent is never shown itself
       for the PARENT_INSTANCES abstracts of p's instances inside the filter, highest score first:
           if not in pool:  final(abstract) = rrf(p), exact ties broken by score(abstract)

5b. THE WALK, when hops is 1 or 2                                             [5] [6]
   frontier = the FRONTIER records of the pool with the highest final score
   for hop = 1 to hops:
       reached = empty
       for each record r in frontier, for each edge of r:
           fact      its cell; its entity's abstract; the chunks that hold its quote
           chunk     the facts whose quotes it holds
           cell      its entity's abstract; the entity's cells before and after it; its facts
           abstract  its entity's cells; its entity's facts (a document's abstract: all its facts)
           parent    the abstracts of the same entity's other instances inside the filter, when parents are on
           for the K_HOP records on that edge with the highest score:
               if not in pool:  value = score * DISCOUNT ^ hop, the highest over the paths that reach it
       frontier = the FRONTIER records of reached with the highest value
   L = the reached records by value, highest first      a third ranked list
   for each record r in L:  final(r) = 1 / (RRF_K + rank of r in L), exact ties broken by score(r)

6. BUNDLE, RENDER AND PACK                                                    [8] [5]
   bundles = for each cell in pool:      the cell, then the pool's facts of the same
                                         entity in the same unit
             for each abstract in pool:  the abstract alone
             the facts in pool whose cell is not in pool, grouped by source passage:
                                         one bundle per passage, headed by its best fact
   a bundle's score is the highest final score among its records

   context = empty    used = 0    tags = empty    previous = none
   for each bundle b, by score, highest first:
       text = date | document | unit, leaving out each one that previous showed
       text = text + the cell's or the abstract's text, when it has one
       for each passage among b's facts, in source order:
           if the passage is in tags:  text = text + "see [Pn] above"
           else:                       text = text + "[Pn]" + the part of the passage not yet shown
           a fact with no passage is shown by its own line
       if used + tokens(text) > budget - tokens(PACKET_HEADER):
           skip b                              a bundle is packed whole or not at all
       else:
           add text to context;  used = used + tokens(text);  add b's new passages to tags
           previous = b's date, document and unit

   source_passage(fact):                       fact -> unit -> raw text in Step 0
       text = the document's text in Step 0
       a, b = the start and end of the fact's unit in text
       if the unit is a chat turn:  passage = text[a : b]
       else:                        passage = the paragraph around the quote, inside a..b
       if the unit is not a chat turn and passage is longer than MAX_PASSAGE:
           passage = the quote and MAX_PASSAGE / 2 characters on either side, inside a..b
       return passage

7. ANSWER                                                                     [6]
   packet = PACKET_HEADER + context
   answer = reader(question, packet, asked on)
```

## A walkthrough: one question, step by step

LongMemEval question gpt4_2ba83207 asks, of one user's chat history: *"Which grocery store did I
spend the most money at in the past month?"* The benchmark's answer is Thrive Market. The user
mentioned three purchases in three different sessions, and the answer needs all three. The numbers
in steps 1 to 4 are from this notebook's log for that question on the 81-document test store, with
20 hits per search.

1. **The filter.** The history is the 53 sessions under `chats/longmemeval/gpt4_2ba83207/`, from
   `path_filter`. Every vector outside those 53 documents is masked out. A parent's vector stays
   only when one of its instances is in the filter.
2. **The vector scan.** The question becomes one 384-number vector, and one matrix product gives
   its cosine with all 15,739 rows. Each record takes the score of its best row, and the best
   records are the vector hits. The best one, at 0.70, is the fact *user went grocery shopping
   at Walmart*.
3. **The keyword scan.** The question's words are joined with OR and searched in the FTS5 index,
   ranked by BM25, inside the filter. The best are the keyword hits. The same Walmart fact is
   first here too.
4. **Fusing the two lists.** Reciprocal rank fusion gives each record 1 / (60 + rank) for every
   list it is in. The Walmart fact is first in both: 1/61 + 1/61 = 0.0328. A fact from the Thrive
   Market session, sixth by vector and eleventh by keyword, gets 1/66 + 1/71 = 0.0292.
5. **The pool.** The hits go into the pool as they are, in fused order. A chat's facts have no
   cells, so here every hit is a fact or a session abstract. A parent hit, when there is one,
   brings in the abstracts of the same entity in this user's other sessions.
6. **Bundling and packing.** With no cells, the facts are grouped by the turn they were read
   from: one bundle per turn, the turn itself read from Step 0 at the facts' unit. Bundles go in
   best first until the next would pass the budget. The Walmart turn comes first, since its fact
   leads both searches, and it is shown as the user's own words in that turn, without the fact
   lines read from it. The date and session print once and are left out of the bundles after it
   until they change.
7. **The reader.** gpt-4o-mini gets the packet (its header, then the bundles), the date the
   question is asked (2023/05/30), and the question, and replies in JSON, brief reasoning first and
   then its answer.

A book uses the rest of the structure. On a novel the hits are cells, facts and abstracts; a cell
is packed with the pool's facts about the same entity in the same chapter, each with its paragraph
of the book.

## The constants

| name | value | what it controls |
|---|---|---|
| `K_ENTRY` | 80 | hits each search returns |
| `HITS` | 40 | hits an arm takes from each search, unless it names more |
| `WIDE_HITS`, `WIDE_PASSAGE` | 80, 2,000 | a wide arm's hits, and the characters of source text it shows with a fact |
| `K_HOP` | 10 | records a hop takes from each edge, the most similar first |
| `FRONTIER` | 20 | records a hop walks on from, the best placed first |
| `DISCOUNT` | 0.8 | a walked record's similarity is multiplied by this for each hop |
| `RRF_K` | 60 | the reciprocal rank fusion constant, as Cormack et al. set it [1] |
| `PARENT_INSTANCES` | 10 | instance abstracts a parent hit brings in, the most similar first |
| `BUDGETS` | 2,000, 12,000 | the grid's budgets: tokens of the packet, header included |
| `FLOOR_LEAST` | 5 | the floor for the best settings keeps the vector hits above the list's mean plus one standard deviation, and at least this many |
| `SEARCHERS`, `SEARCH_BATCH` | 4, 32 | keyword searches run at once on their own connections, for a batch of questions before their contexts |
| `RERANKER`, `RERANK_CANDIDATES` | ms-marco-MiniLM-L-6-v2, 100 | the cross-encoder a reranked arm orders its pool by (Nogueira and Cho 2019), and the chunks or turns a reranked flat arm scores |
| `OMNI_ITEMS`, `OMNI_READER` | 19, gpt-4.1-mini | OmniMemEval's 20 items (the header and 19 bundles) and its reader |
| `MAX_PASSAGE` | 500 | characters of source text given with one fact; a longer passage is cut to the quote and 250 either side |

Each is logged with every question, with the version, the embedder and the arm's settings.

## The settings

Retrieval runs with a **setting**: hops (0, 1 or 2), a budget (2,000 or 12,000 tokens), the filter
on or off, the search fused or vector alone, and parents on or off. Every benchmark runs the grid
of hops by budget, filtered and, where the store holds more than the benchmark's documents,
unfiltered. Each question is searched once and every setting is built from that search. A
setting's time is that search plus its own walk and packing. Each benchmark's best setting, by the
benchmark's own figure, then runs widened (80 hits, 2,000-character passages), with the vector
search alone, with parents off, in date order, with short answers and with the floor; the floor
also runs on the best setting at 2,000 tokens. On LongMemEval the best setting also runs with no
filter, read by gpt-4o, and as OmniMemEval asks: 20 items, on our reader and on gpt-4.1-mini.
`ONLY` runs one benchmark alone, so each can have its own kernel.

## The evaluation

| measurement | set | arms | scored by |
|---|---|---|---|
| Chat memory | LongMemEval, all 500 questions | the grid, filtered to the question's history; flat retrieval over raw turns at each budget; full context; the best setting with the vector search alone, with parents off, with no filter, and read by `gpt-4o-2024-08-06` | LongMemEval's own judge, `gpt-4o-2024-08-06` |
| Novels | GraphRAG-Bench, 2,010 questions | the grid filtered to the question's own novel and unfiltered over the whole store; plain RAG with the benchmark's chunks and the store's model at each budget and at the benchmark's 5 chunks, filtered and over all 20 novels; no context; the best setting with the vector search alone and with parents off | the benchmark's own code, run from a pinned commit |
| Medical | GraphRAG-Bench, 2,062 questions | on the Medical store, filtered to its one document: the grid, plain RAG at each budget and at 5 chunks, no context, and the best setting with the vector search alone and with parents off | the benchmark's own code, run from a pinned commit |
| Bodies of work | the 189 books and papers | the model's groups against abstract vectors at .30 | the source folders |
| Instruments | the whole build | none | counted from the logs |

The Oz key and the model's bodies of work are scored at the end of the global-layer notebook, and
this notebook prints those scores beside its own. Five questions a person checks by eye (Tip
becoming Ozma, Watson's wound and others) are printed, never scored.

```
load the store and the two benchmarks' questions

for each benchmark the store holds (LongMemEval, the novels, Medical), for each arm of it:
    for each question the arm has not answered:
        context = what the arm gives the reader
            store arm:   retrieve(question, filter or none, its setting),
                         every store arm of a question from the same two searches
            flat arm:    the raw text most similar to the question, until the arm's limit
            full arm:    every session of the history, newest first, up to 120,000 tokens
            no context:  nothing
        save the context
    for each saved context not yet answered, 16 at a time:
        answer = reader(question, context, date asked)
        if memory: verdict = the benchmark's judge(question, gold answer, answer)
        save them
    stop between questions when the session or the spend runs out

run GraphRAG-Bench's evaluator over each arm's answers
each benchmark's best setting: run it with the vector search alone and with parents off
    (LongMemEval: also with no filter and on gpt-4o), and score those arms
score the bodies of work against the .30 baseline
print the eyeball questions, the instruments and the tables
```

Each context and answer is written as it is made. When a session stops, the next session attaches
this notebook's output, copies it back, and goes on from the next question.

## What every arm shares

Blocks 2 to 8 are one retrieval path: the reader `gpt-4o-mini-2024-07-18` at temperature 0 unless
an arm names another, `PACKET_HEADER` and `ANSWER_PROMPT`, `HITS` 40, `RRF_K` 60,
`PARENT_INSTANCES` 10, `MAX_PASSAGE` 500. Every arm of both benchmarks uses this prompt and passes
through `ask_reader`. The path uses one regular expression, in `keyword_entries`.

## Versions

- 0.21: chunks are records, found by both searches, walked to and from their facts, and shown as
  numbered sources; the chunks-only arm runs them alone through the same search; the evaluator runs
  as the benchmark publishes it, with no change.
- 0.20: the grid of hops 0, 1 and 2 by budgets of 2,000 and 12,000, filtered and not, from one
  search per question; the walk's records are a third ranked list, fused by rank; each benchmark's
  best setting gets its own arms; plain RAG fills each budget, each chunk named; the reader's
  guidance heads the packet; chat turns are shown whole; the reader is pinned to `gpt-4o-mini-2024-07-18`; the evaluator runs 8 at a time
  and scores retrieval for the `quality` arms only.
- 0.19: the pool is the hits with no walk; a parent hit brings its instances' abstracts; 40 hits per
  search; a fact's line is left out when its passage is shown; a header leaves out what the bundle
  before it showed; a passage is at most 500 characters.
- 0.18: plain RAG's chunks are embedded with the store's `bge-small`.
- 0.17: the reader gives brief reasoning before its answer.
- 0.16: a bundle scores as its best record; the bodies baseline cuts at distance .30; misses are
  reported by the turns LongMemEval marks; tests are on the same reader; retrieval is timed alone.
- 0.15: 20 facts are taken from each top entity.
- 0.14: each top entity's best facts are taken, and a passage's text is shown once.
- 0.13: the evaluator keeps a faithfulness verdict with no `reason`.
- 0.12: the evaluator's environment is built with virtualenv.
- 0.11: two LongMemEval arms are read by gpt-4o.
- 0.10: a fact marked `inverse` prints the way it was read; the reader no longer sees LongMemEval's
  `answer_` and `_abs` marks.

## Running it

On Kaggle: attach the global-layer notebook's output, the two Step datasets and the raw corpus, add the
`OPENAI_API_KEY` secret, turn Internet on, and start it from the editor. The first session embeds
the chat turns. Set `FIRST_QUESTIONS` to 10
to try every arm on ten questions before the full run. When a session stops, attach its output and
run again. Locally: set `STORE_DIR`, `STEP0_DIR`, `STEP1_DIR`, `BENCHMARKS` and `OUT`.

## Block 0: the key and the connection

Run this first. It looks for `OPENAI_API_KEY` (the environment, then the Kaggle secret of that
name) and makes one tiny call to the reader model. It prints one of three things: the key works;
there is no key, in which case retrieval still runs and the reader does not; or the key or the
connection failed, with OpenAI's own message, and the run stops here.

In [ ]:
import json
import os
import urllib.error
import urllib.request

READER = "gpt-4o-mini-2024-07-18"
STRONGER_READER = "gpt-4o-2024-08-06"     # LongMemEval's best setting read again by gpt-4o


def find_key():
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("OPENAI_API_KEY")
    except Exception:
        return None


KEY = find_key()

if not KEY:
    print("NO KEY: OPENAI_API_KEY is not in the environment or attached as a Kaggle secret.")
    print("The run stops before the first arm.")
    print("On Kaggle: Add-ons, Secrets, tick OPENAI_API_KEY, then Save Version, Save & Run All.")
else:
    payload = {
        "model": READER,
        "temperature": 0,
        "response_format": {"type": "json_object"},
        "messages": [{"role": "user", "content": 'Reply with the JSON object {"ok": true}.'}],
    }
    request = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": "Bearer " + KEY, "Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            body = json.loads(response.read().decode("utf-8"))
        print("KEY FOUND, CONNECTION WORKS:", body["model"], "replied", body["choices"][0]["message"]["content"])
    except urllib.error.HTTPError as error:
        print("KEY FOUND, BUT OPENAI REFUSED THE CALL: HTTP", error.code)
        print(error.read().decode("utf-8", errors="replace")[:500])
        raise SystemExit("fix the key or the account, then run again")
    except urllib.error.URLError as error:
        print("KEY FOUND, BUT NO CONNECTION TO OPENAI:", error.reason)
        raise SystemExit("turn Internet on in the notebook's settings, then run again")

## Block 1: files and settings

Where the store, the text and the benchmarks are, where the output goes, and the constants. The
retrieval constants are the retrieval notebook's. An earlier session's output is copied back here.

```
find the store, Step 0 and Step 1 (Kaggle's mounts, or the environment)
if this session's output is empty and an earlier session is attached: copy it back
set the retrieval constants, then this notebook's own
```

In [ ]:
import hashlib
import json
import math
import os
import queue
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np

VERSION = "threadatlas-retrieval 0.21"

try:
    import fastembed
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastembed"], check=True)
try:
    import tiktoken
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")


def first_with(file, folders):
    """The first of these folders that holds this file."""
    for folder in folders:
        if (folder / file).exists():
            return folder
    return None


def attached_with(file):
    """The attached folder holding the largest copy of this file: the latest session's."""
    found = None
    largest = -1
    if Path("/kaggle/input").exists():
        for path in Path("/kaggle/input").rglob(file):
            size = path.stat().st_size
            if size > largest:
                found = path.parent
                largest = size
    return found


STORE_DIR = first_with("threadatlas.sqlite", [Path(os.environ.get("STORE_DIR", "data/global"))])
if STORE_DIR is None:
    STORE_DIR = attached_with("threadatlas.sqlite")
if STORE_DIR is None:
    raise SystemExit("threadatlas.sqlite not found: attach the global-layer notebook's output")

# any other attached store is opened too: the Medical store, built from GraphRAG-Bench's medical corpus alone
MORE_STORE_DIRS = []
if Path("/kaggle/input").exists():
    for path in sorted(Path("/kaggle/input").rglob("threadatlas.sqlite")):
        if path.parent != STORE_DIR:
            MORE_STORE_DIRS.append(path.parent)
if os.environ.get("MORE_STORE_DIR"):
    MORE_STORE_DIRS.append(Path(os.environ["MORE_STORE_DIR"]))

# Step 0, the documents' text: the public dataset for the main store, the Medical build's output for the Medical store
STEP0_DIRS = []
for folder in [Path(os.environ.get("STEP0_DIR", "data/step0")), Path(os.environ.get("MORE_STEP0_DIR", "data/step0-more")),
               Path("/kaggle/input/datasets/jhffmn/it494-threadatlas-step0"), Path("/kaggle/input/it494-threadatlas-step0"),
               Path("/kaggle/input/notebooks/jhffmn/threadatlas-medical/export"), Path("/kaggle/input/threadatlas-medical/export"),
               Path("/kaggle/input/datasets/jhffmn/threadatlas-medical-chunks/export"),
               Path("/kaggle/input/threadatlas-medical-chunks/export")]:
    if (folder / "documents.jsonl").exists() and folder not in STEP0_DIRS:
        STEP0_DIRS.append(folder)
if len(STEP0_DIRS) == 0:
    raise SystemExit("documents.jsonl not found: attach the it494-threadatlas-step0 dataset")

STEP1_DIR = first_with("receipt.json", [Path(os.environ.get("STEP1_DIR", "data/step1")),
                                        Path("/kaggle/input/datasets/jhffmn/it494-threadatlas-step1"),
                                        Path("/kaggle/input/it494-threadatlas-step1"),
                                        Path("/kaggle/input/notebooks/jhffmn/threadatlas-medical/packages"),
                                        Path("/kaggle/input/threadatlas-medical/packages")])
if STEP1_DIR is None:
    raise SystemExit("receipt.json not found: attach the it494-threadatlas-step1 dataset")

OUT = Path(os.environ.get("OUT", "/kaggle/working"))
BENCHMARKS = Path(os.environ.get("BENCHMARKS", str(OUT / "benchmarks")))
CONTEXTS = OUT / "contexts"
ANSWERS = OUT / "answers"
SCORES = OUT / "scores"
FLAT = OUT / "flat"


def restore_run():
    """Kaggle's working folder starts empty, so an attached earlier session is copied back."""
    if (OUT / "calls.jsonl").exists() or not Path("/kaggle/input").exists():
        return
    earlier = None
    longest = -1
    for path in Path("/kaggle/input").rglob("questions.jsonl"):       # the longest log of this version is the latest
        with open(path, encoding="utf-8") as f:
            first = f.readline()
        try:
            version = json.loads(first).get("constants", {}).get("VERSION")
        except ValueError:
            version = None
        if version != VERSION:
            continue
        size = path.stat().st_size
        if size > longest:
            earlier = path.parent
            longest = size
    if earlier is not None:
        shutil.copytree(earlier, OUT, dirs_exist_ok=True)
        print(f"going on from {earlier}")


restore_run()
for folder in (BENCHMARKS, CONTEXTS, ANSWERS, SCORES, FLAT):
    folder.mkdir(parents=True, exist_ok=True)

# the retrieval constants
MODEL = "BAAI/bge-small-en-v1.5"
DIMENSION = 384

K_ENTRY = 80                 # hits each search returns; an arm takes its first `hits`
HITS = 40                    # hits an arm takes from each search, unless it names more
WIDE_HITS = 80               # hits a wide arm takes
WIDE_PASSAGE = 2000          # characters of source text a wide arm shows with a fact
K_HOP = 10                   # records a hop takes from each edge, the most similar first
FRONTIER = 20                # records a hop walks on from, the best placed first
DISCOUNT = 0.8               # a walked record's similarity is multiplied by this for each hop
RRF_K = 60                   # the reciprocal rank fusion constant
BUDGET = 12000               # tokens of context the reader is given, when an arm names no budget
BUDGETS = (2000, 12000)      # the budgets of the grid
FLOOR_LEAST = 5              # the floor keeps the vector hits above the list's mean plus one standard deviation,
                             # and at least this many
OMNI_ITEMS = 19              # OmniMemEval returns 20 items: the packet's header and this many bundles
OMNI_READER = "gpt-4.1-mini"     # OmniMemEval's reader
RERANKER = "Xenova/ms-marco-MiniLM-L-6-v2"     # the cross-encoder a reranked arm orders its records by
RERANK_CANDIDATES = 100      # the raw chunks or turns a reranked flat arm scores, the most similar first
QUERY_PREFIX = ""
MAX_PASSAGE = 500            # characters of source text shown with one fact, centred on its quote
PARENT_INSTANCES = 10        # instance abstracts a parent hit brings in, the most similar first

# this notebook's own
FIRST_QUESTIONS = None       # set to 10 to try every arm on ten questions of each set
ONLY = ("novels", "medical")  # "memory", "novels" or "medical", or several in a tuple, to run those alone; None runs every one
FILTERED_ONLY = True         # True runs a GraphRAG-Bench set's filtered arms alone, False its unfiltered ones; None both
ARMS_ONLY = ("hop1-12000",)   # names of the only GraphRAG-Bench arms to run, such as ("hop1-12000",); None runs every arm
BEST_ARMS = False            # False leaves out each benchmark's extra arms on its best setting (Block 14b)
WORKERS = 16                 # reader calls in flight at once
SEARCHERS = 4                # keyword searches run at once, each on its own connection to the store
SEARCH_BATCH = 32            # questions whose keyword searches are made together, before their contexts
SCORERS = 8                  # evaluator processes at once
FULL_WORKERS = 4             # the full-context arm sends 120,000 tokens a call
TIME_STOP_HOURS = 8          # a session stops itself here, between questions
SPEND_STOP = 200.0           # dollars on the reader and the judge, counted over every session of a run
MAX_FULL_TOKENS = 120000     # the full-context arm's limit, under gpt-4o-mini's 128,000
FLAT_LOOK = 1000             # rows a flat arm reads down before it gives up filling the budget
HAND_CHECK = 30              # judged answers drawn for reading by hand
SEED = 20261101

# plain RAG: the chunks of GraphRAG-Bench's paper (appendix H.2) and the hard-coded overlap of its code,
# embedded with the store's own MODEL, so plain RAG and the system differ only in raw text against the graph
CHUNK_TOKENS = 256
CHUNK_OVERLAP = 32
TOP_CHUNKS = 5

# one commit of the benchmark's repository, for its evaluator and its files, and their hashes;
# the evaluator's embedding model for answer correctness, as the benchmark's README sets it
LARGE_MODEL = "BAAI/bge-large-en-v1.5"
SCORER_MODEL = "gpt-4o-mini"      # the evaluator's model, as its README runs it
SCORER_COMMIT = "fdbab5959b18c96532580877ffe27d112bccc0ec"
SCORER_URL = "https://codeload.github.com/GraphRAG-Bench/GraphRAG-Benchmark/tar.gz/" + SCORER_COMMIT
RAW_URL = "https://raw.githubusercontent.com/GraphRAG-Bench/GraphRAG-Benchmark/" + SCORER_COMMIT
# GraphRAG-Bench's two sets; each runs when the store holds its documents
GRAPHRAG_SETS = {
    "novels": {"questions": "novel_questions.json", "questions_sha": "d02b3db9c44d273371960f700f9ff9bf174dbc885542a7cadf636cded32c7af1",
               "corpus": "novel.json", "corpus_sha": "b0b4b2e7dc1b5d72f774783130d08b5d62e922fa9dc3e58612154b78b6089d15"},
    "medical": {"questions": "medical_questions.json", "questions_sha": "ca6ffc4217c892777bd3ba3009e5539716f3b3213aec9c1b04476cdf7c5fb620",
                "corpus": "medical.json", "corpus_sha": "e15c3b8af5931610c005cbf465e58c2b9ff3b9fa83b7c4bfb042bd8343cdf002"},
}
MEMORY_URL = "https://huggingface.co/datasets/xiaowu0162/longmemeval/resolve/main/longmemeval_s"
MEMORY_SHA = "08d8dad4be43ee2049a22ff5674eb86725d0ce5ff434cde2627e5e8e7e117894"

STARTED = time.time()


class StopRun(Exception):
    """The session stops here, on time or on spend, and the next one goes on."""


print(VERSION)
print("store: ", STORE_DIR / "threadatlas.sqlite")
for folder in MORE_STORE_DIRS:
    print("store: ", folder / "threadatlas.sqlite")
for folder in STEP0_DIRS:
    print("text:  ", folder / "documents.jsonl")
print("step 1:", STEP1_DIR)
print("output:", OUT)

## Block 2: opening the store

The store and its vectors, checked against each other, and the lookups every question uses:
which record each vector row belongs to, which documents each parent reaches, each document's
title and date, and each entity's name. Then the filter: the documents under a path, such as one
LongMemEval history's folder. Last, the source text: each of the store's documents from
Step 0, and the start and end of each of its units. Every attached store is opened the same way;
`store` is the main one, and `STORES` holds it and any other.

In [ ]:
def open_store(folder):
    loading = time.time()
    db = sqlite3.connect(folder / "threadatlas.sqlite")
    vectors = np.load(folder / "threadatlas.npy")

    header = db.execute("select model, dimension from vec_header").fetchone()
    row_count = db.execute("select count(*) from vec_row").fetchone()[0]
    if header != (MODEL, DIMENSION) or vectors.shape != (row_count, DIMENSION):
        raise SystemExit(f"the vectors do not match the store: {header}, {vectors.shape}, {row_count} rows")

    record_of_row = []
    rows_of_record = {}
    for row, kind, doc_id, record_id in db.execute("select row, record, doc_id, record_id from vec_row order by row"):
        record = (kind, doc_id, record_id)
        record_of_row.append(record)
        if record not in rows_of_record:
            rows_of_record[record] = []
        rows_of_record[record].append(row)

    documents_of_parent = {}
    for doc_id, parent_id in db.execute("select doc_id, parent_id from instance_of"):
        parent_id = str(parent_id)
        if parent_id not in documents_of_parent:
            documents_of_parent[parent_id] = set()
        documents_of_parent[parent_id].add(doc_id)

    documents = {}
    for doc_id, title, source_uri, occurred_at in db.execute("select doc_id, title, source_uri, occurred_at from document"):
        documents[doc_id] = {"title": title, "source_uri": source_uri, "occurred_at": occurred_at}

    entity_names = {}
    document_node = {}
    for doc_id, node_id, name, kind in db.execute("select doc_id, node_id, name, kind from node"):
        entity_names[(doc_id, node_id)] = name
        if kind == "document":
            document_node[doc_id] = node_id

    unit_position = {}
    for unit_id, position in db.execute("select unit_id, position from unit"):
        unit_position[unit_id] = position

    # a chunk's offsets, and the facts each chunk holds; a store built before chunks has none
    chunks = {}
    facts_in_chunk = {}
    chunks_of_fact = {}
    if db.execute("select count(*) from sqlite_master where name = 'chunk'").fetchone()[0] > 0:
        for doc_id, chunk_id, start, end in db.execute("select doc_id, chunk_id, start, end from chunk"):
            chunks[("chunk", doc_id, chunk_id)] = (start, end)
        for doc_id, fact_id, chunk_id in db.execute("select doc_id, fact_id, chunk_id from fact_in_chunk"):
            facts_in_chunk.setdefault(("chunk", doc_id, chunk_id), []).append(("fact", doc_id, fact_id))
            chunks_of_fact.setdefault(("fact", doc_id, fact_id), []).append(("chunk", doc_id, chunk_id))

    # each vector row's document, or its parent, as numbers, so a filter is one array operation
    doc_index = {}
    for doc_id in documents:
        doc_index[doc_id] = len(doc_index)
    row_doc = np.full(len(record_of_row), -1, dtype=np.int64)
    row_parent = np.full(len(record_of_row), -1, dtype=np.int64)
    for row in range(len(record_of_row)):
        kind, doc_id, record_id = record_of_row[row]
        if kind == "parent":
            row_parent[row] = int(record_id)
        else:
            row_doc[row] = doc_index[doc_id]
    parents_of_doc = {}
    for parent_id in documents_of_parent:
        for doc_id in documents_of_parent[parent_id]:
            if doc_id not in parents_of_doc:
                parents_of_doc[doc_id] = set()
            parents_of_doc[doc_id].add(int(parent_id))

    store = {}
    store["db"] = db
    store["path"] = folder / "threadatlas.sqlite"
    store["vectors"] = vectors.astype(np.float32)
    store["record_of_row"] = record_of_row
    store["rows_of_record"] = rows_of_record
    store["documents_of_parent"] = documents_of_parent
    store["documents"] = documents
    store["entity_names"] = entity_names
    store["document_node"] = document_node
    store["unit_position"] = unit_position
    store["doc_index"] = doc_index
    store["row_doc"] = row_doc
    store["row_parent"] = row_parent
    store["parents_of_doc"] = parents_of_doc
    store["chunks"] = chunks
    store["facts_in_chunk"] = facts_in_chunk
    store["chunks_of_fact"] = chunks_of_fact
    chunk_rows = np.zeros(len(record_of_row), dtype=bool)
    for row in range(len(record_of_row)):
        if record_of_row[row][0] == "chunk":
            chunk_rows[row] = True
    store["chunk_rows"] = chunk_rows
    print(f"{folder.name}: lookups read in {time.time() - loading:.0f} seconds; {len(chunks)} chunks")
    store["graph"] = read_graph(db, unit_position)
    print(f"{folder.name}: graph read in {time.time() - loading:.0f} seconds")
    return store


def read_graph(db, unit_position):
    """What a walk steps along, read once: each entity's facts, cells and abstract, each fact's entity and unit, each parent's instances."""
    graph = {"fact_of": {}, "facts_of_node": {}, "facts_of_doc": {}, "facts_of_cell": {}, "cells_of_node": {},
             "abstracts": set(), "parent_of": {}, "instances": {}}
    for doc_id, fact_id, subject, unit_id in db.execute("select doc_id, fact_id, subject, unit_id from fact"):
        graph["fact_of"][(doc_id, fact_id)] = (subject, unit_id)
        graph["facts_of_node"].setdefault((doc_id, subject), []).append(fact_id)
        graph["facts_of_doc"].setdefault(doc_id, []).append(fact_id)
        graph["facts_of_cell"].setdefault((doc_id, subject, unit_id), []).append(fact_id)
    placed = {}
    for doc_id, node_id, unit_id in db.execute("select doc_id, node_id, unit_id from cell"):
        placed.setdefault((doc_id, node_id), []).append((unit_position.get(unit_id, 0), unit_id))
    for key in placed:
        placed[key].sort()
        units = []
        for position, unit_id in placed[key]:
            units.append(unit_id)
        graph["cells_of_node"][key] = units
    for doc_id, node_id in db.execute("select doc_id, node_id from abstract"):
        graph["abstracts"].add((doc_id, node_id))
    for doc_id, node_id, parent_id in db.execute("select doc_id, node_id, parent_id from instance_of"):
        graph["parent_of"][(doc_id, node_id)] = parent_id
        graph["instances"].setdefault(parent_id, []).append((doc_id, node_id))
    return graph


def path_filter(store, path):
    doc_ids = set()
    for doc_id, source_uri in store["db"].execute("select doc_id, source_uri from document"):
        if path in source_uri:
            doc_ids.add(doc_id)
    return doc_ids


def read_source(store, step0_dirs):
    """Each of the store's documents and units from whichever Step 0 folder holds them."""
    texts = {}
    unit_bounds = {}
    for step0 in step0_dirs:
        with open(step0 / "documents.jsonl", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                if row["doc_id"] in store["documents"]:
                    texts[row["doc_id"]] = row["text"]
        with open(step0 / "units.jsonl", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                if row["doc_id"] in store["documents"]:
                    unit_bounds[row["unit_id"]] = (row["start"], row["end"])

    missing = 0
    for doc_id in store["documents"]:
        if doc_id not in texts:
            missing += 1
    if missing > 0:
        raise SystemExit(f"{missing} documents in the store have no text in Step 0")

    store["texts"] = texts
    store["unit_bounds"] = unit_bounds


def warm(path):
    """The file read through once, so the full passes below come from memory, not the slow input disk."""
    started = time.time()
    with open(path, "rb") as f:
        while f.read(1 << 24):
            pass
    print(f"{path.name}: read through in {time.time() - started:.0f} seconds")


STORES = []
for folder in [STORE_DIR] + MORE_STORE_DIRS:
    warm(folder / "threadatlas.sqlite")
    warm(folder / "threadatlas.npy")
    opened = open_store(folder)
    read_source(opened, STEP0_DIRS)
    STORES.append(opened)
    print(folder.name + ":", len(opened["documents"]), "documents,", len(opened["record_of_row"]), "vectors,",
          len(opened["unit_bounds"]), "units of text from Step 0")
store = STORES[0]

## Block 3: scoring the question against every vector

The question's vector, its similarity to every row, and a record's score as its best row.
`best_first` is the one sort used everywhere: highest score first, ties broken by the record's
name.

In [ ]:
EMBEDDER = None


def embed(texts):
    global EMBEDDER
    if EMBEDDER is None:
        from fastembed import TextEmbedding
        EMBEDDER = TextEmbedding(MODEL)
    vectors = list(EMBEDDER.embed(texts, batch_size=256))
    return np.asarray(vectors, dtype=np.float32)


def score_every_row(store, question):
    question_vector = embed([QUERY_PREFIX + question])[0]
    return store["vectors"] @ question_vector


embed(["load the model before any question is timed"])


def record_score(store, row_scores, record):
    rows = store["rows_of_record"].get(record)
    if not rows:
        return 0.0
    return float(row_scores[rows].max())


def best_first(records, score_of):
    ranked = []
    for record in records:
        name = []
        for part in record:
            name.append(str(part))
        ranked.append((-score_of[record], tuple(name), record))
    ranked.sort()

    ordered = []
    for item in ranked:
        ordered.append(item[2])
    return ordered

## Block 4: the two entry scans

Steps 1 to 4 of the walkthrough: the filter, the vector scan, the keyword scan, and the fusion.

In [ ]:
def rows_in_filter(store, doc_filter, use_parents):
    """The vector rows a search may return: rows of documents in the filter, and parents with an instance in it."""
    if doc_filter is None:
        keep = store["row_parent"] < 0                    # every document row
        if use_parents:
            keep = np.ones(len(store["record_of_row"]), dtype=bool)
        return keep
    wanted = []
    parents = set()
    for doc_id in doc_filter:
        wanted.append(store["doc_index"][doc_id])
        parents |= store["parents_of_doc"].get(doc_id, set())
    keep = np.isin(store["row_doc"], wanted)
    if use_parents and len(parents) > 0:
        keep = keep | np.isin(store["row_parent"], list(parents))
    return keep


def vector_entries(store, row_scores, keep):
    scores = np.where(keep, row_scores, -np.inf)
    best = {}
    for row in np.argsort(-scores, kind="stable"):
        score = float(scores[row])
        if score == -np.inf:
            break
        if len(best) >= K_ENTRY and score < min(best.values()):
            break
        record = store["record_of_row"][row]
        if record not in best:
            best[record] = score
    return best_first(best, best)[:K_ENTRY]


def keyword_entries(store, question, doc_filter, db=None, kind=None):
    words = re.findall(r"\w+", question)
    if len(words) == 0:
        return []
    quoted = []
    for word in words:
        quoted.append('"' + word + '"')
    match = " OR ".join(quoted)

    sql = "select record, doc_id, record_id from search where search match ?"
    args = [match]
    if kind is not None:                                   # one kind of record: the chunks-only arm
        sql += " and record = ?"
        args.append(kind)
    if doc_filter is not None:
        marks = ", ".join(["?"] * len(doc_filter))
        sql += " and doc_id in (" + marks + ")"
        args += sorted(doc_filter)
    sql += " order by bm25(search), record, doc_id, record_id limit ?"
    args.append(K_ENTRY)

    found = []
    if db is None:
        db = store["db"]
    for row in db.execute(sql, args):
        found.append(tuple(row))
    return found


KEYWORDS = {}          # keyword hits made ahead of their question, with the seconds each search took
CONNECTIONS = {}


def connections(store):
    """SEARCHERS connections to the store, opened once and kept, so their caches stay warm across batches."""
    key = str(store["path"])
    with LOCK:
        if key not in CONNECTIONS:
            CONNECTIONS[key] = queue.Queue()
            for i in range(SEARCHERS):
                CONNECTIONS[key].put(sqlite3.connect(store["path"], check_same_thread=False))
        return CONNECTIONS[key]


def keyword_job(store, question, doc_filter, key):
    started = time.time()
    free = connections(store)
    db = free.get()                                        # one search on a connection at a time
    try:
        found = keyword_entries(store, question, doc_filter, db)
    finally:
        free.put(db)
    KEYWORDS[key] = (found, time.time() - started)


def prefetch_keywords(store_arms, items):
    """The keyword searches every store arm of these questions will make, SEARCHERS at a time."""
    filters = set()
    for arm in store_arms:
        filters.add(arm["filtered"])
    jobs = []
    for item in items:
        store = STORES[item["store"]]
        for filtered in filters:
            doc_filter = None
            if filtered:
                doc_filter = path_filter(store, item["path"])
            key = (id(store), item["question"], doc_filter is None)
            if key not in KEYWORDS:
                jobs.append((store, item["question"], doc_filter, key))
    in_parallel(keyword_job, jobs, SEARCHERS)


def fuse(vector_ranked, keyword_ranked):
    rrf = {}
    for ranked in [vector_ranked, keyword_ranked]:
        rank = 1
        for record in ranked:
            rrf[record] = rrf.get(record, 0.0) + 1 / (RRF_K + rank)
            rank += 1
    return best_first(rrf, rrf), rrf

## Block 5: the pool, and the walk

Step 5. The pool starts as the hits themselves. Every entry that is a fact, a cell or an abstract
goes in as it is. A parent is never shown to the reader, so a parent hit
brings in what it stands for: the abstracts of its instances inside the filter, which are the same
entity in other documents, the `PARENT_INSTANCES` most similar first. Every record keeps the fused
rank of the hit that brought it in, and the similarity of its own best sentence breaks ties.

```
pool = empty
for each entry e that is not a parent:
    pool[e] = rrf(e), ties broken by score(e)
for each entry p that is a parent:
    for the PARENT_INSTANCES abstracts of p's instances inside the filter with the highest score:
        if the abstract is not in pool:  pool[abstract] = rrf(p), ties broken by its own score
```

Step 5b, for a setting with hops. The walk starts from the `FRONTIER` best-placed records of the
pool. Each step takes, along each edge, the `K_HOP` records most similar to the question that are
not in the pool; a record found this way is valued at its similarity times `DISCOUNT` for each hop,
the best over the paths that reach it. The second hop walks on from the `FRONTIER` records the
first one reached with the highest values. The records the walk reached form a third ranked list,
by value, and each is placed at 1 / (`RRF_K` + its rank in that list), the rule the two searches
are fused by. With hops 0 the pool is the hits and what the parent hits bring in.

In [ ]:
def abstract_above(store, doc_id, node_id):
    abstracts = store["graph"]["abstracts"]
    if (doc_id, node_id) in abstracts:
        return [("abstract", doc_id, node_id)]

    # an entity with no abstract of its own (a chat's) is routed to its document's abstract
    document = store["document_node"].get(doc_id)
    if document and (doc_id, document) in abstracts:
        return [("abstract", doc_id, document)]
    return []


def instance_abstracts(store, parent_id, doc_filter, skip=None):
    """The abstracts of a parent's instances inside the filter, leaving out the instance a walk came from."""
    abstracts = []
    for doc_id, node_id in store["graph"]["instances"].get(parent_id, []):
        if (doc_id, node_id) == skip:
            continue
        if doc_filter is not None and doc_id not in doc_filter:
            continue
        abstracts += abstract_above(store, doc_id, node_id)
    return abstracts


def facts_of(store, doc_id, node_id):
    """An entity's facts; a document's own node has every fact of its document."""
    graph = store["graph"]
    fact_ids = graph["facts_of_node"].get((doc_id, node_id), [])
    if node_id == store["document_node"].get(doc_id):
        fact_ids = graph["facts_of_doc"].get(doc_id, [])
    facts = []
    for fact_id in fact_ids:
        facts.append(("fact", doc_id, fact_id))
    return facts


def edges_of(store, record, doc_filter, use_parents):
    """The records one step from this one, by edge."""
    graph = store["graph"]
    kind, doc_id, record_id = record
    if kind == "chunk":                             # a chunk leads to the facts read from its text
        return {"facts": list(store["facts_in_chunk"].get(record, []))}
    edges = {}
    if kind == "fact":
        node_id, unit_id = graph["fact_of"][(doc_id, record_id)]
        edges["cell"] = []
        if unit_id in graph["cells_of_node"].get((doc_id, node_id), []):
            edges["cell"] = [("cell", doc_id, node_id + "@" + unit_id)]
        edges["abstract"] = abstract_above(store, doc_id, node_id)
        edges["chunk"] = list(store["chunks_of_fact"].get(record, []))
    elif kind == "cell":
        node_id, unit_id = record_id.split("@")
        units = graph["cells_of_node"].get((doc_id, node_id), [])
        here = units.index(unit_id)
        edges["abstract"] = abstract_above(store, doc_id, node_id)
        edges["previous"] = []
        if here > 0:
            edges["previous"] = [("cell", doc_id, node_id + "@" + units[here - 1])]
        edges["next"] = []
        if here + 1 < len(units):
            edges["next"] = [("cell", doc_id, node_id + "@" + units[here + 1])]
        edges["facts"] = []
        for fact_id in graph["facts_of_cell"].get((doc_id, node_id, unit_id), []):
            edges["facts"].append(("fact", doc_id, fact_id))
    else:
        node_id = record_id
        edges["cells"] = []
        for unit_id in graph["cells_of_node"].get((doc_id, node_id), []):
            edges["cells"].append(("cell", doc_id, node_id + "@" + unit_id))
        edges["facts"] = facts_of(store, doc_id, node_id)
    if use_parents and (doc_id, node_id) in graph["parent_of"]:
        edges["parent"] = instance_abstracts(store, graph["parent_of"][(doc_id, node_id)], doc_filter, (doc_id, node_id))
    return edges


def walk(store, row_scores, pool, doc_filter, hops, use_parents):
    """Step 5b: hops steps through the graph from the pool's best-placed records; what they reach is one more
    ranked list, by similarity times DISCOUNT for each hop, fused with the two searches by rank."""
    frontier = best_first(pool, score_of(pool))[:FRONTIER]
    walked = {}
    values = {}
    for hop in range(1, hops + 1):
        reached = {}
        for record in frontier:
            edges = edges_of(store, record, doc_filter, use_parents)
            for edge in edges:
                candidates = set(edges[edge])
                candidates.discard(record)
                scores = {}
                for candidate in candidates:
                    scores[candidate] = record_score(store, row_scores, candidate)
                for candidate in best_first(candidates, scores)[:K_HOP]:
                    if candidate in pool:
                        continue
                    value = scores[candidate] * DISCOUNT ** hop
                    if candidate in values and values[candidate] >= value:
                        continue
                    walked[candidate] = {"hops": hop, "edge": edge, "from": list(record), "similarity": scores[candidate]}
                    values[candidate] = value
                    reached[candidate] = value
        frontier = best_first(reached, reached)[:FRONTIER]
    rank = 1
    for record in best_first(walked, values):
        place = 1 / (RRF_K + rank)
        item = walked[record]
        pool[record] = {"hops": item["hops"], "edge": item["edge"], "from": item["from"], "base": place,
                        "score": ranked_score(place, item["similarity"])}
        rank += 1


def score_of(pool):
    scores = {}
    for record in pool:
        scores[record] = pool[record]["score"]
    return scores


def ranked_score(rrf_value, similarity):
    """A record's place in the pool: its fused rank, with its own similarity deciding only exact ties."""
    return rrf_value + 1e-12 * similarity


def make_pool(store, row_scores, entries, rrf, doc_filter):
    """Step 5: the hits, and for each parent hit the abstracts of its instances."""
    pool = {}
    for entry in entries:
        if entry[0] != "parent":
            similarity = record_score(store, row_scores, entry)
            pool[entry] = {"hops": 0, "edge": "entry", "from": None, "base": rrf[entry],
                           "score": ranked_score(rrf[entry], similarity)}

    # a parent hit stands for its instances, so it brings in their abstracts
    for entry in entries:
        if entry[0] != "parent":
            continue
        candidates = set(instance_abstracts(store, int(entry[2]), doc_filter))
        scores = {}
        for record in candidates:
            scores[record] = record_score(store, row_scores, record)
        for record in best_first(candidates, scores)[:PARENT_INSTANCES]:
            if record not in pool:
                pool[record] = {"hops": 1, "edge": "children", "from": list(entry), "base": rrf[entry],
                                "score": ranked_score(rrf[entry], scores[record])}
    return pool

## Block 6: bundles, and what the reader sees

Step 6. The pool is grouped into bundles: a cell with the pool's facts of the same entity in the
same unit; an abstract alone; and the facts whose cell is not in the pool, one bundle for each
passage of the source they were read from. A bundle is one dated block of text: the cell's or the
abstract's text, then the passages its facts were read from. A fact whose passage is shown prints no
line of its own; a fact with no passage prints its line. A bundle's passages that overlap are joined
into one span of the source. A passage wanted a second time is not repeated; it carries a number,
`[P3]`, and the later bundle points to it. A passage that overlaps one already shown prints only its
new text. Every cut falls between words. A bundle's header gives its date, document and unit, and
leaves out each one the bundle before it already showed. A bundle scores as its best record, and
bundles are packed whole to the token budget. A document whose title is missing or is its file
name, as a chat's is, is shown by the first 8 characters of its `doc_id`.

In [ ]:
TOKENIZER = None


def count_tokens(text):
    global TOKENIZER
    if TOKENIZER is None:
        TOKENIZER = tiktoken.get_encoding("o200k_base")
    return len(TOKENIZER.encode(text, disallowed_special=()))       # a chat can hold the words <|endoftext|>


def document_title(store, doc_id):
    document = store["documents"][doc_id]
    title = document["title"]
    if title is None or title == Path(document["source_uri"]).stem:
        return doc_id[:8]
    return title


def unit_label_and_date(db, unit_id):
    found = db.execute("select label, occurred_at from unit where unit_id = ?", (unit_id,)).fetchone()
    if found is None:
        return None, None
    return found[0], found[1]


def whole_words(text, start, end):
    """A cut moved inward to the nearest space at each end, so no word is split."""
    while start < end and start > 0 and not text[start - 1].isspace() and not text[start].isspace():
        start += 1
    while end > start and end < len(text) and not text[end].isspace() and not text[end - 1].isspace():
        end -= 1
    return start, end


def source_passage(store, doc_id, unit_id, quote_start, quote_end):
    text = store["texts"][doc_id]
    unit_start, unit_end = store["unit_bounds"][unit_id]
    kind = store["db"].execute("select kind from unit where unit_id = ?", (unit_id,)).fetchone()[0]

    start = unit_start
    end = unit_end
    if kind != "user" and kind != "assistant":
        for blank_line in ["\n\n", "\r\n\r\n"]:
            found = text.rfind(blank_line, unit_start, quote_start)
            if found != -1 and found > start:
                start = found
            found = text.find(blank_line, quote_end, unit_end)
            if found != -1 and found < end:
                end = found

    if end - start > MAX_PASSAGE and kind != "user" and kind != "assistant":
        start = max(unit_start, quote_start - MAX_PASSAGE // 2)
        end = min(unit_end, quote_end + MAX_PASSAGE // 2)
        start, end = whole_words(text, start, end)
    return start, end, text[start:end].strip()


def read_fact(store, doc_id, fact_id):
    fields = ["subject", "subject_name", "predicate", "object", "object_is_node",
              "qualifiers", "occurred_at", "direction", "quote", "quote_start", "quote_end", "unit_id"]
    values = store["db"].execute("select " + ", ".join(fields) + " from fact where doc_id = ? and fact_id = ?",
                                 (doc_id, fact_id)).fetchone()
    fact = {}
    for i in range(len(fields)):
        fact[fields[i]] = values[i]
    return fact


def fact_line(store, doc_id, fact):
    names = store["entity_names"]
    subject = names.get((doc_id, fact["subject"]), fact["subject"])
    if fact["direction"] == "mentioned" and fact["subject_name"]:
        subject = fact["subject_name"]
    thing = fact["object"]
    if fact["object_is_node"]:
        thing = names.get((doc_id, fact["object"]), fact["object"])
    if fact["direction"] == "inverse":                        # stored under the major its object points at
        subject, thing = thing, subject
    line = subject + " " + fact["predicate"].replace("_", " ") + " " + thing
    if fact["qualifiers"]:
        line += " (" + fact["qualifiers"] + ")"
    return line


def passage_of(store, doc_id, fact):
    if not fact["quote"]:
        return None, None
    start, end, words = source_passage(store, doc_id, fact["unit_id"], fact["quote_start"], fact["quote_end"])
    return (doc_id, start, end), words


def make_bundles(store, pool):
    cell_of = {}
    for record in pool:
        if record[0] == "cell":
            node_id, unit_id = record[2].split("@")
            cell_of[(record[1], node_id, unit_id)] = record

    bundles = {}
    for record in pool:
        if record[0] != "fact":
            bundles[record] = []

    # a fact goes under its cell when the cell is in the pool; the other facts are grouped by passage
    groups = {}
    for record in pool:
        if record[0] != "fact":
            continue
        fact = read_fact(store, record[1], record[2])
        place = (fact["quote_start"] or 0, record[2], record)
        cell = cell_of.get((record[1], fact["subject"], fact["unit_id"]))
        if cell is not None:
            bundles[cell].append(place)
            continue
        passage, words = passage_of(store, record[1], fact)
        if passage is None:
            passage = record
        if passage not in groups:
            groups[passage] = []
        groups[passage].append(place)

    # a group is headed by its best-scoring fact
    for passage in groups:
        scores = {}
        for place in groups[passage]:
            scores[place[2]] = pool[place[2]]["score"]
        head = best_first(scores, scores)[0]
        bundles[head] = groups[passage]

    for head in bundles:
        bundles[head].sort()
        facts = []
        for place in bundles[head]:
            facts.append(place[2])
        bundles[head] = facts
    return bundles


def fresh_words(store, passage, tags, new):
    """A passage's text not shown yet, and the number of the passage that covers it when none is left."""
    doc_id, start, end = passage
    shown = []
    for key in tags:
        if key[0] == doc_id:
            shown.append((key[1], key[2], tags[key]))
    place = len(tags)
    for key in new:
        place += 1
        if key[0] == doc_id:
            shown.append((key[1], key[2], place))
    for first, last, number in shown:
        if first <= start and end <= last:
            return "", number
        if first <= start < last:
            start = last
        if first < end <= last:
            end = first
    start, end = whole_words(store["texts"][doc_id], start, end)
    if end <= start:                         # covered from both sides by passages already shown
        return "", shown[0][2]
    return store["texts"][doc_id][start:end].strip(), None


def fact_lines(store, doc_id, facts, tags, indent):
    by_passage = {}
    order = []
    texts = {}
    for record in facts:
        fact = read_fact(store, doc_id, record[2])
        passage, words = passage_of(store, doc_id, fact)
        if passage is None:
            passage = record
        else:
            texts[passage] = words
        if passage not in by_passage:
            by_passage[passage] = []
            order.append(passage)
        by_passage[passage].append(fact_line(store, doc_id, fact))

    # a fact with no quote is shown by its line alone
    block = ""
    for passage in order:
        if passage not in texts:
            for line in by_passage[passage]:
                block += "\n" + indent + "- " + line

    # the bundle's passages in source order, each run of overlapping ones joined into one span
    spans = []
    for passage in order:
        if passage not in texts:
            continue
        if len(spans) > 0 and spans[-1][0] == passage[0] and passage[1] <= spans[-1][2]:
            spans[-1] = (passage[0], spans[-1][1], max(spans[-1][2], passage[2]))
        else:
            spans.append(passage)

    new = []
    for passage in spans:
        if passage in tags:
            block += "\n" + indent + "  source: see [P" + str(tags[passage]) + "] above"
            continue
        words, covered = fresh_words(store, passage, tags, new)
        if covered is not None:
            block += "\n" + indent + "  source: see [P" + str(covered) + "] above"
        else:
            new.append(passage)
            number = len(tags) + len(new)
            block += "\n" + indent + "  source [P" + str(number) + ']: "' + words + '"'
    return block, new


def header(date, title, label, previous):
    """A bundle's date, document and unit, leaving out each one the bundle before it already showed."""
    pieces = []
    if previous is None or previous[0] != date:
        pieces.append(str(date))
    if previous is None or previous[1] != title:
        pieces.append(str(title))
    if previous is None or previous[2] != label:
        pieces.append(str(label))
    if len(pieces) == 0:
        return ""
    return " | ".join(pieces) + " | "


def render(store, head, facts, tags, previous):
    """A bundle's text, its new passages, and its date, document and unit for the next bundle's header."""
    db = store["db"]
    names = store["entity_names"]
    kind, doc_id, record_id = head
    title = document_title(store, doc_id)
    document_date = store["documents"][doc_id]["occurred_at"]

    if kind == "abstract":
        text = db.execute("select text from abstract where doc_id = ? and node_id = ?", (doc_id, record_id)).fetchone()[0]
        entity = names.get((doc_id, record_id), record_id)
        shown = (document_date, title, "abstract")
        return header(shown[0], shown[1], shown[2], previous) + f"{entity}: {text}", [], shown

    if kind == "fact":
        fact = read_fact(store, doc_id, record_id)
        label, unit_date = unit_label_and_date(db, fact["unit_id"])
        shown = (fact["occurred_at"] or unit_date or document_date, title, label or "record")
        lines, new = fact_lines(store, doc_id, facts, tags, "  ")
        return header(shown[0], shown[1], shown[2], previous).removesuffix(" | ") + lines, new, shown

    if kind == "chunk":
        # a chunk is shown as a numbered source, so a fact read from inside it points to it rather than
        # repeating its words, and only the part of it not shown yet is printed
        start, end = store["chunks"][head]
        passage = (doc_id, start, end)
        shown = (document_date, title, "chunk " + record_id)
        top = header(shown[0], shown[1], shown[2], previous).removesuffix(" | ")
        if passage in tags:
            return top + "\n  source: see [P" + str(tags[passage]) + "] above", [], shown
        words, covered = fresh_words(store, passage, tags, [])
        if covered is not None:
            return top + "\n  source: see [P" + str(covered) + "] above", [], shown
        return top + "\n  source [P" + str(len(tags) + 1) + ']: "' + words + '"', [passage], shown

    node_id, unit_id = record_id.split("@")
    text = db.execute("select text from cell where doc_id = ? and node_id = ? and unit_id = ?",
                      (doc_id, node_id, unit_id)).fetchone()[0]
    label, unit_date = unit_label_and_date(db, unit_id)
    entity = names.get((doc_id, node_id), node_id)
    shown = (unit_date or document_date, title, label or unit_id[:8])
    lines, new = fact_lines(store, doc_id, facts, tags, "  ")
    return header(shown[0], shown[1], shown[2], previous) + f"{entity}: {text}" + lines, new, shown


def place_in_time(store, head, shown):
    """A bundle's place in date order: its date, document and unit."""
    unit_id = None
    if head[0] == "fact":
        unit_id = read_fact(store, head[1], head[2])["unit_id"]
    elif head[0] == "cell":
        unit_id = head[2].split("@")[1]
    return (str(shown[0] or ""), str(shown[1]), store["unit_position"].get(unit_id, -1), list(head))


def in_date_order(store, pool, bundles, chosen, budget):
    """The chosen bundles rendered again in date order; the lowest-scored is dropped until they fit."""
    while len(chosen) > 0:
        context = []
        used = 0
        tags = {}
        previous = None
        for head in sorted(chosen, key=chosen.get):
            text, new, shown = render(store, head, bundles[head], tags, previous)
            context.append(text)
            used += count_tokens(text)
            previous = shown
            for passage in new:
                tags[passage] = len(tags) + 1
        if used <= budget:
            return context, used
        lowest = None
        for head in chosen:
            if lowest is None or pool[head]["score"] < pool[lowest]["score"]:
                lowest = head
        del chosen[lowest]
        pool[lowest]["packed"] = False
        for record in bundles[lowest]:
            pool[record]["packed"] = False
    return [], 0


def pack(store, pool, budget, order="score", items=None):
    bundles = make_bundles(store, pool)
    scores = {}
    for head in bundles:
        scores[head] = pool[head]["score"]
        for record in bundles[head]:
            if pool[record]["score"] > scores[head]:
                scores[head] = pool[record]["score"]

    context = []
    used = 0
    tags = {}
    previous = None
    chosen = {}
    for head in best_first(bundles, scores):
        text, new, shown = render(store, head, bundles[head], tags, previous)
        size = count_tokens(text)
        packed = used + size <= budget
        if items is not None and len(chosen) >= items:          # an item budget: at most this many bundles
            packed = False
        pool[head]["bundle"] = list(head)
        pool[head]["packed"] = packed
        for record in bundles[head]:
            pool[record]["bundle"] = list(head)
            pool[record]["packed"] = packed
        if not packed:
            continue
        context.append(text)
        used += size
        previous = shown
        chosen[head] = place_in_time(store, head, shown)
        for passage in new:
            tags[passage] = len(tags) + 1
    if order == "date":
        return in_date_order(store, pool, bundles, chosen, budget)
    return context, used

## Block 7: the whole path, and its log

`retrieve` runs steps 1 to 6 and writes one line to `questions.jsonl`. `keyword`, `vector` and
`parent` turn each part off; `hops` and `budget` set the rest of the setting.

In [ ]:
def rank_in(ranked, record):
    if record in ranked:
        return ranked.index(record) + 1
    return None


def search(store, question, doc_filter, parent, searches):
    """Both scans at K_ENTRY, made once for a question and its filter and shared by every approach."""
    key = (id(store), question, doc_filter is None, parent)
    if searches is not None and key in searches:
        return searches[key]
    started = time.time()
    row_scores = score_every_row(store, question)
    keep = rows_in_filter(store, doc_filter, parent)
    ahead = KEYWORDS.get((id(store), question, doc_filter is None))
    if ahead is None:
        keyword = keyword_entries(store, question, doc_filter)
        ahead_seconds = 0.0
    else:
        keyword, ahead_seconds = ahead                     # made ahead on another connection; its time still counts
    found = {"row_scores": row_scores, "vector": vector_entries(store, row_scores, keep),
             "keyword": keyword, "seconds": time.time() - started + ahead_seconds}
    if searches is not None:
        searches[key] = found
    return found


def above_floor(store, row_scores, ranked):
    """The vector hits the floor keeps: those above the list's mean plus one standard deviation, at least FLOOR_LEAST."""
    scores = []
    for record in ranked:
        scores.append(record_score(store, row_scores, record))
    mean = sum(scores) / len(scores)
    spread = 0.0
    for value in scores:
        spread += (value - mean) ** 2
    deviation = (spread / len(scores)) ** 0.5
    keep = FLOOR_LEAST
    for i in range(len(scores)):
        if scores[i] >= mean + deviation:
            keep = max(keep, i + 1)
    return ranked[:keep]


CROSS_ENCODER = None


def cross_scores(question, texts):
    """The cross-encoder's score of the question with each text."""
    global CROSS_ENCODER
    if CROSS_ENCODER is None:
        from fastembed.rerank.cross_encoder import TextCrossEncoder
        CROSS_ENCODER = TextCrossEncoder(RERANKER)
    scores = []
    for value in CROSS_ENCODER.rerank(question, texts, batch_size=64):
        scores.append(float(value))
    return scores


def record_text(store, record):
    """A record's own words, as the reranker reads them: a fact's line and quote, a cell's or an abstract's text."""
    kind, doc_id, record_id = record
    db = store["db"]
    if kind == "fact":
        fact = read_fact(store, doc_id, record_id)
        return fact_line(store, doc_id, fact) + " " + (fact["quote"] or "")
    if kind == "cell":
        node_id, unit_id = record_id.split("@")
        return db.execute("select text from cell where doc_id = ? and node_id = ? and unit_id = ?",
                          (doc_id, node_id, unit_id)).fetchone()[0]
    if kind == "chunk":
        start, end = store["chunks"][record]
        return store["texts"][doc_id][start:end]
    return db.execute("select text from abstract where doc_id = ? and node_id = ?", (doc_id, record_id)).fetchone()[0]


def rerank_pool(store, question, pool):
    """Every record in the pool placed by the cross-encoder's score; its fused place breaks exact ties."""
    records = list(pool)
    texts = []
    for record in records:
        texts.append(record_text(store, record))
    scores = cross_scores(question, texts)
    for i in range(len(records)):
        pool[records[i]]["score"] = scores[i] + 1e-9 * pool[records[i]]["score"]


def reranked_rows(question, index, scores, order):
    """A flat arm's rows in the cross-encoder's order: the RERANK_CANDIDATES most similar, scored with the question."""
    rows = []
    texts = []
    for row in order[:RERANK_CANDIDATES].tolist():
        if scores[row] == -np.inf:
            break
        rows.append(row)
        texts.append(index["rows"][row][2])
    by_cross = cross_scores(question, texts)
    ranked = []
    for i in range(len(rows)):
        ranked.append((-by_cross[i], i, rows[i]))
    ranked.sort()
    ordered = []
    for item in ranked:
        ordered.append(item[2])
    return ordered


def chunk_search(store, question, doc_filter, row_scores):
    """Both searches over the store's chunks alone: the chunks-only arm's own two lists."""
    keep = rows_in_filter(store, doc_filter, False) & store["chunk_rows"]
    return vector_entries(store, row_scores, keep), keyword_entries(store, question, doc_filter, kind="chunk")


def retrieve(store, question, doc_filter=None, keyword=True, vector=True, parent=True, budget=BUDGET,
             hops=0, hits=HITS, searches=None, floor=False, order="score", items=None, passage=None, rerank=False,
             chunks_only=False):
    found = search(store, question, doc_filter, parent, searches)
    started = time.time()                                  # this approach's own work starts after the search
    row_scores = found["row_scores"]
    found_vector = found["vector"]
    found_keyword = found["keyword"]
    if chunks_only:
        found_vector, found_keyword = chunk_search(store, question, doc_filter, row_scores)

    vector_ranked = []
    if vector:
        vector_ranked = found_vector[:hits]                # the first hits of a ranked list are that search at hits
    if floor and len(vector_ranked) > 0:
        vector_ranked = above_floor(store, row_scores, vector_ranked)
    keyword_ranked = []
    if keyword:
        keyword_ranked = found_keyword[:hits]
    entries, rrf = fuse(vector_ranked, keyword_ranked)

    pool = make_pool(store, row_scores, entries, rrf, doc_filter)
    if hops > 0:
        walk(store, row_scores, pool, doc_filter, hops, parent)
    if rerank and len(pool) > 0:
        rerank_pool(store, question, pool)
    global MAX_PASSAGE
    usual = MAX_PASSAGE
    if passage is not None:
        MAX_PASSAGE = passage                          # one question at a time, so the width is restored before the next
    try:
        context, used = pack(store, pool, budget, order, items)
    finally:
        MAX_PASSAGE = usual

    entry_log = []
    for record in entries:
        entry_log.append({"record": list(record), "rank_v": rank_in(vector_ranked, record),
                          "rank_w": rank_in(keyword_ranked, record), "rrf": round(rrf[record], 5)})
    pool_log = []
    for record in pool:
        item = pool[record]
        pool_log.append({"record": list(record), "hops": item["hops"], "edge": item["edge"], "from": item["from"],
                         "score": round(item["score"], 4), "bundle": item["bundle"], "packed": item["packed"]})
    filter_log = None
    if doc_filter:
        filter_log = sorted(doc_filter)

    line = {
        "asked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "question": question,
        "filter": filter_log,
        "arms": {"keyword": keyword, "vector": vector, "parent": parent, "budget": budget, "hops": hops, "hits": hits,
                 "floor": floor, "order": order, "items": items, "passage": passage, "rerank": rerank,
                 "chunks_only": chunks_only},
        "constants": {"VERSION": VERSION, "MODEL": MODEL, "K_ENTRY": K_ENTRY, "RRF_K": RRF_K,
                      "prefix": QUERY_PREFIX, "MAX_PASSAGE": MAX_PASSAGE, "PARENT_INSTANCES": PARENT_INSTANCES,
                      "K_HOP": K_HOP, "FRONTIER": FRONTIER, "DISCOUNT": DISCOUNT},
        "entries": entry_log,
        "pool": pool_log,
        "context": context,
        "tokens": used,
        "seconds": round(found["seconds"] + time.time() - started, 3),     # the shared search counts for every approach
    }
    with open(OUT / "questions.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
    return context, line

## Block 8: the reader

Step 7: one call to the reader, gpt-4o-mini at temperature 0, with the key Block 0 found, retried
when the server is busy; the answer and the call's cost are written to `calls.jsonl`. Then the
judge: LongMemEval's own, its prompts copied word for word from the benchmark's evaluator
(`src/evaluation/evaluate_qa.py`), one prompt per question type, on `gpt-4o-2024-08-06` at
temperature 0. A reply containing "yes" means correct. `READER` in Block 0 is the one place the
reader is set. When the date a question is asked is known, the reader is told it. The prompt tells
the reader that a later record is current when two disagree, to say so when a fact asked for is not
in the records, and to build advice on what the records say about the user.

In [ ]:
PRICES = {"gpt-4o-mini": (0.15, 0.60), "gpt-4o-mini-2024-07-18": (0.15, 0.60),
          "gpt-4o-2024-08-06": (2.50, 10.00), "gpt-4.1-mini": (0.40, 1.60)}      # $ per million tokens in, out

# the head of every retrieval packet, so any reader is told how to read the records under it
PACKET_HEADER = """The records below are dated and name their documents; a "source" is the passage of the
document its facts were read from. When records disagree, the one with the later date is current. If the
question asks for a fact the records do not state, say so. If it asks for advice or a recommendation, give
one, built on what the records say about the person asking: what they own, use and prefer."""

ANSWER_PROMPT = """{header}

Records:
{context}
{today}
Question: {question}

Answer from the records.{style} Reply with a JSON object: {{"reasoning": "<brief reasoning from the records>", "answer": "<your answer>"}}."""

CONCISE = " Keep the answer short: what is asked, without restating the question or the records."


SPENT = 0.0


def post(payload):
    request = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": "Bearer " + KEY, "Content-Type": "application/json"},
    )
    for attempt in range(3):
        try:
            with urllib.request.urlopen(request, timeout=900) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as error:
            busy = error.code == 429 or error.code >= 500
            if not busy or attempt == 2:
                raise
            time.sleep(15 * (attempt + 1))


def plain(text):
    """A text without the marks LongMemEval puts in session ids: answer_ on gold, _abs on an unanswerable history's."""
    text = text.replace("answer_", "")
    for end in ("_", " ", ":", ",", ".", ")", "\n"):           # the end of an id, so a book's _absolute_ stays
        text = text.replace("_abs" + end, end)
    return text


def ask_reader(question, context, asked_on=None, model=READER, concise=False):
    global SPENT
    today = ""
    if asked_on:
        today = "\nThe question is asked on " + asked_on + ".\n"
    style = ""
    if concise:
        style = CONCISE
    prompt = plain(ANSWER_PROMPT.format(header=PACKET_HEADER, context="\n".join(context), today=today, question=question,
                                        style=style))
    payload = {
        "model": model,
        "temperature": 0,
        "response_format": {"type": "json_object"},
        "messages": [{"role": "user", "content": prompt}],
    }
    started = time.time()
    body = post(payload)

    try:
        answer = json.loads(body["choices"][0]["message"]["content"])["answer"]
    except (ValueError, KeyError, TypeError):
        answer = None

    usage = body.get("usage", {})
    price_in, price_out = PRICES[model]
    cost = (usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1e6
    SPENT += cost
    call = {"stage": "reader", "question": question, "asked_on": asked_on, "answer": answer, "model": body.get("model"),
            "in": usage.get("prompt_tokens"), "out": usage.get("completion_tokens"),
            "seconds": round(time.time() - started, 1), "cost": cost}
    with open(OUT / "calls.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(call, ensure_ascii=False) + "\n")
    return answer, call["seconds"]


# LongMemEval's own judge: the prompts of src/evaluation/evaluate_qa.py in the benchmark's repository,
# word for word, on the model it names, at temperature 0 with 10 tokens; a reply containing "yes" is correct
JUDGE = "gpt-4o-2024-08-06"

CHECK_PLAIN = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
CHECK_TEMPORAL = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. In addition, do not penalize off-by-one errors for the number of days. If the question asks for the number of days/weeks/months, etc., and the model makes off-by-one errors (e.g., predicting 19 days when the answer is 18), the model's response is still correct. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
CHECK_UPDATE = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response contains some previous information along with an updated answer, the response should be considered as correct as long as the updated answer is the required answer.\n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
CHECK_PREFERENCE = "I will give you a question, a rubric for desired personalized response, and a response from a model. Please answer yes if the response satisfies the desired response. Otherwise, answer no. The model does not need to reflect all the points in the rubric. The response is correct as long as it recalls and utilizes the user's personal information correctly.\n\nQuestion: {}\n\nRubric: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."

CHECKS = {
    "single-session-user": CHECK_PLAIN,
    "single-session-assistant": CHECK_PLAIN,
    "multi-session": CHECK_PLAIN,
    "temporal-reasoning": CHECK_TEMPORAL,
    "knowledge-update": CHECK_UPDATE,
    "single-session-preference": CHECK_PREFERENCE,
}


def judge(question_type, question, correct_answer, response):
    global SPENT
    prompt = CHECKS[question_type].format(question, correct_answer, response)
    payload = {"model": JUDGE, "temperature": 0, "max_tokens": 10,
               "messages": [{"role": "user", "content": prompt}]}
    body = post(payload)
    reply = body["choices"][0]["message"]["content"].strip()

    usage = body.get("usage", {})
    price_in, price_out = PRICES[JUDGE]
    cost = (usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1e6
    SPENT += cost
    call = {"stage": "judge", "question": question, "reply": reply, "model": body.get("model"),
            "in": usage.get("prompt_tokens"), "out": usage.get("completion_tokens"), "cost": cost}
    with open(OUT / "calls.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(call, ensure_ascii=False) + "\n")
    return "yes" in reply.lower()


if KEY:
    print("reader", READER, "at temperature 0, judge", JUDGE, "; key present")
else:
    print("reader", READER, "; no key")

## Block 9: the questions

LongMemEval's `_s` file and GraphRAG-Bench's question file, each checked against its hash. An
unanswerable question, one whose id holds `_abs`, goes to the benchmark's fifth prompt, added here.
Each question carries
the path of the documents it may search: its history's folder, or its source's file. A benchmark
runs only when an attached store holds its documents, and each question searches the store that
holds them: LongMemEval and the novels the main store, Medical the store built from GraphRAG-Bench's
medical corpus alone. A LongMemEval turn marked as holding the answer is used only to report
whether the context showed it.

```
open each benchmark file, attached or downloaded, and check its SHA-256
memory = each LongMemEval question
    its check is its type, or "abstention" if "_abs" is in its id
    its path is chats/longmemeval/<id>/
    its marked turns are the turns of its answer sessions that LongMemEval marks as holding the answer
for each GraphRAG-Bench set, novels and medical:
    each question, its path graphrag-bench/<source>.txt
keep the questions whose path is in a store, each marked with that store; a benchmark with none is skipped
```

In [ ]:
CHECK_ABSTENTION = "I will give you an unanswerable question, an explanation, and a response from a model. Please answer yes if the model correctly identifies the question as unanswerable. The model could say that the information is incomplete, or some other information is given but the asked information is not.\n\nQuestion: {}\n\nExplanation: {}\n\nModel Response: {}\n\nDoes the model correctly identify the question as unanswerable? Answer yes or no only."

CHECKS["abstention"] = CHECK_ABSTENTION


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        block = f.read(1 << 22)
        while block:
            digest.update(block)
            block = f.read(1 << 22)
    return digest.hexdigest()


def benchmark_file(name, url, sha256):
    """A benchmark file, attached or downloaded, and proven to be the file these numbers name."""
    path = BENCHMARKS / name
    attached = attached_with(name)
    if not path.exists() and attached is not None:
        path = attached / name
    if not path.exists():
        path = BENCHMARKS / name
        print("downloading", name)
        urllib.request.urlretrieve(url, path)
    if file_hash(path) != sha256:
        raise SystemExit(f"{name} is not the file this harness was built on")
    return json.loads(path.read_text(encoding="utf-8"))


def marked_turns(row):
    """The turns LongMemEval marks as holding the answer, with their whitespace closed up."""
    turns = []
    for session_id, session in zip(row["haystack_session_ids"], row["haystack_sessions"]):
        if session_id in row["answer_session_ids"]:
            for turn in session:
                if turn.get("has_answer"):
                    turns.append(" ".join(turn["content"].split()))
    return turns


def shows_turn(text, turn):
    """True when the whole turn stands in the text, spacing aside."""
    return " ".join(turn.split()) in text


def in_store(items, stores):
    """The questions whose documents are in one of these stores, each marked with the first that holds them."""
    found = {}
    kept = []
    for item in items:
        if item["path"] not in found:
            found[item["path"]] = None
            for number in range(len(stores)):
                if found[item["path"]] is None and len(path_filter(stores[number], item["path"])) > 0:
                    found[item["path"]] = number
        if found[item["path"]] is not None:
            item["store"] = found[item["path"]]
            kept.append(item)
    return kept


def holds(path):
    """True when any attached store holds a document under this path."""
    for opened in STORES:
        if len(path_filter(opened, path)) > 0:
            return True
    return False


def read_memory():
    """LongMemEval's 500 questions."""
    if len(path_filter(store, "chats/longmemeval/")) == 0:      # the chat histories are in the main store
        return []
    items = []
    for row in benchmark_file("longmemeval_s.json", MEMORY_URL, MEMORY_SHA):
        check = row["question_type"]
        if "_abs" in row["question_id"]:
            check = "abstention"
        items.append({"id": row["question_id"], "type": row["question_type"], "check": check,
                      "question": row["question"], "answer": str(row["answer"]),
                      "asked_on": row["question_date"], "path": "chats/longmemeval/" + row["question_id"] + "/",
                      "marked_turns": marked_turns(row)})
    return items


def read_graphrag(set_name):
    """One GraphRAG-Bench set's questions, each searching its own source document."""
    files = GRAPHRAG_SETS[set_name]
    url = RAW_URL + "/Datasets/Questions/" + files["questions"]
    items = []
    for row in benchmark_file(files["questions"], url, files["questions_sha"]):
        items.append({"id": row["id"], "set": set_name, "type": row["question_type"], "check": None,
                      "question": row["question"], "answer": row["answer"], "evidence": row["evidence"],
                      "asked_on": None, "path": "graphrag-bench/" + row["source"] + ".txt",
                      "source": row["source"]})
    return items


def first_of(items):
    """The whole set, or a seeded sample when trying the arms."""
    if FIRST_QUESTIONS is None:
        return items
    return random.Random(SEED).sample(items, FIRST_QUESTIONS)


MEMORY = first_of(in_store(read_memory(), [store]))
print(len(MEMORY), "LongMemEval questions")

GRAPHRAG = {}
if holds("graphrag-bench/"):          # stores with no GraphRAG-Bench document download nothing
    for set_name in GRAPHRAG_SETS:
        items = first_of(in_store(read_graphrag(set_name), STORES))
        if len(items) > 0:
            GRAPHRAG[set_name] = items
        print(len(items), "GraphRAG-Bench", set_name, "questions")

if ONLY is not None:
    wanted = ONLY
    if not isinstance(ONLY, (list, tuple)):
        wanted = (ONLY,)
    if "memory" not in wanted:
        MEMORY = []
    for set_name in list(GRAPHRAG):
        if set_name not in wanted:
            del GRAPHRAG[set_name]
    print("this kernel runs", ", ".join(wanted), "alone")

## Block 10: the arms

An arm is what the reader is given. A store arm is one setting: `retrieve` with its hops, its
budget, and its `keyword`, `vector` and `parent` switches. `grid` builds the
arms of hops 0, 1 and 2 at each budget of `BUDGETS`. `quality` marks the arms whose retrieval the GraphRAG-Bench evaluator also
scores; every arm's answers are scored. Turning the parent off also takes parent rows out of the
entry scan. The arms run in the order they are listed.

```
an arm = { name, how (store, flat, full or none), filter on or off, parent on or off,
           keyword on or off, vector on or off, hops, hits, budget, reader, quality }
```

In [ ]:
def store_arm(name, filtered=True, parent=True, keyword=True, vector=True, budget=BUDGET, reader=READER,
              hops=0, hits=HITS, quality=False, floor=False, order="score", concise=False, items=None,
              passage=None, rerank=False, chunks_only=False):
    return {"name": name, "how": "store", "filtered": filtered, "parent": parent, "keyword": keyword,
            "vector": vector, "budget": budget, "reader": reader, "hops": hops, "hits": hits, "quality": quality,
            "floor": floor, "order": order, "concise": concise, "items": items, "passage": passage, "rerank": rerank,
            "chunks_only": chunks_only}


def other_arm(name, how, filtered=True, budget=BUDGET, reader=READER, quality=False, concise=False, rerank=False):
    return {"name": name, "how": how, "filtered": filtered, "budget": budget, "reader": reader, "quality": quality,
            "concise": concise, "rerank": rerank}


def grid(prefix, filtered):
    """Hops 0, 1 and 2 at each budget of BUDGETS; retrieval quality is scored at the smallest."""
    arms = []
    for hops in (0, 1, 2):
        for budget in BUDGETS:
            arms.append(store_arm(prefix + "hop" + str(hops) + "-" + str(budget), filtered=filtered, hops=hops,
                                  budget=budget, quality=filtered and budget == BUDGETS[0]))
    return arms


def plain_arms(how, prefix, filtered):
    arms = []
    for budget in BUDGETS:
        arms.append(other_arm(prefix + str(budget), how, filtered=filtered, budget=budget,
                              quality=how == "chunks" and filtered and budget == BUDGETS[0]))
    return arms


def hybrid_arms(prefix, filtered, budgets=BUDGETS):
    """The store's chunks alone, through the same fused search, with no graph record and no walk: what the
    search adds over plain RAG, so the graph's own share is the gap from here to the full store."""
    arms = []
    for budget in budgets:
        arms.append(store_arm(prefix + "chunks-hybrid-" + str(budget), filtered=filtered, budget=budget, hops=0,
                              parent=False, chunks_only=True))
    return arms


EXTRAS = {}           # each benchmark's arms on its best setting, added once the grid is scored (Block 14b)


def arms_for(set_name):
    """A set's arms. When the store holds only this set's documents, no filter is the filter, so those arms are left out."""
    held = STORES[GRAPHRAG[set_name][0]["store"]]
    inside = set()
    for item in GRAPHRAG[set_name]:
        inside |= path_filter(held, item["path"])
    arms = []
    for arm in NOVEL_ARMS + EXTRAS.get(set_name, []):
        if not arm["filtered"] and len(inside) == len(held["documents"]):
            continue
        if FILTERED_ONLY is not None and arm["filtered"] != FILTERED_ONLY:
            continue                          # a kernel that runs one half of the set's arms
        if ARMS_ONLY is not None and arm["name"] not in ARMS_ONLY:
            continue                          # a kernel that runs only the named arms, such as a submission
        arms.append(arm)
    return arms


def unfiltered_grid():
    """The whole store searched, no filter: hops 0 and 2 at each budget, and hops 1 at the largest."""
    arms = []
    for budget in BUDGETS:
        for hops in (0, 2):
            arms.append(store_arm("no-filter-hop" + str(hops) + "-" + str(budget), filtered=False, hops=hops,
                                  budget=budget))
    arms.append(store_arm("no-filter-hop1-" + str(BUDGETS[-1]), filtered=False, hops=1, budget=BUDGETS[-1]))
    return arms


# every benchmark is run filtered and unfiltered, each beside its raw-text baseline and the chunks alone
MEMORY_ARMS = (grid("", True) + hybrid_arms("", True) + plain_arms("turns", "flat-", True)
               + unfiltered_grid() + hybrid_arms("no-filter-", False) + plain_arms("turns", "flat-no-filter-", False)
               + [other_arm("full-context", "full")])

NOVEL_ARMS = (grid("", True) + hybrid_arms("", True) + plain_arms("chunks", "plain-rag-", True)
              + unfiltered_grid() + hybrid_arms("no-filter-", False) + plain_arms("chunks", "plain-rag-no-filter-", False)
              + [other_arm("plain-rag", "chunks", budget=None), other_arm("no-context", "none")])

## Block 11: the flat baselines

The baselines read raw text and have vectors of their own. LongMemEval's is one vector per chat
turn with `bge-small`, the store's model and embedder, searched by cosine and packed whole to the same budget as
the system, each turn dated the way a record is. A turn repeated across histories is embedded once
and shown once. GraphRAG-Bench's is its own plain RAG: 256-token chunks with 32 tokens of overlap,
embedded with `bge-small` like the store, each named by its document and its number in it; an arm
takes the top 5, as the benchmark's paper does, or fills its budget.

The turns, about 247,000 of them, are embedded on a GPU when there is one. Vectors are saved
20,000 at a time.

```
turn index:
    for each chat unit: text = the document's text from its start to its end, without its header
    vectors of the distinct texts, 20,000 at a time, each batch saved
    one row per turn, pointing at its text's vector
chunk index:
    for each novel: cut its tokens into 256 with 32 shared, decode, embed with bge-small
flat search(question, filter):
    score every row, drop rows outside the filter, best first
```

In [ ]:
BATCH = 20000


def encode_turns(texts):
    """A batch of turns, by the store's embedder, the one every question goes through."""
    return embed(texts)


FLAT_ENCODER = {"name": "fastembed"}       # what made the turn vectors; the flat baseline's questions use the same
FULL_MODEL = None


def full_precision(texts):
    """The same model at full precision through sentence-transformers, normalised as the store's vectors are."""
    global FULL_MODEL
    if FULL_MODEL is None:
        try:
            import sentence_transformers
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
        from sentence_transformers import SentenceTransformer
        FULL_MODEL = SentenceTransformer(MODEL, device="cpu")
    return np.asarray(FULL_MODEL.encode(texts, normalize_embeddings=True, show_progress_bar=False), dtype=np.float32)


def flat_question(question):
    """A question's vector for the flat baseline, by the encoder that made its turn vectors."""
    if FLAT_ENCODER["name"] == "full":
        return full_precision([QUERY_PREFIX + question])[0]
    return embed([QUERY_PREFIX + question])[0]


def borrow_turns(texts):
    """Turn vectors an attached earlier run made, copied in when they hold exactly these texts; the first texts are
    embedded again by each encoder to find which one made them, and none is used if neither matches."""
    if not Path("/kaggle/input").exists() or (FLAT / "turns-000.npy").exists():
        return
    for first in sorted(Path("/kaggle/input").rglob("turns-000.npy")):
        files = sorted(first.parent.glob("turns-*.npy"))
        rows = 0
        for path in files:
            rows += np.load(path, mmap_mode="r").shape[0]
        if rows != len(texts):
            print("turn vectors in", first.parent, "hold", rows, "rows, not", len(texts), "; not used")
            continue
        stored = np.load(files[0])[:8].astype(np.float32)
        sample = texts[:8]
        quantized = float(np.mean(np.sum(stored * embed(sample), axis=1)))
        full = float(np.mean(np.sum(stored * full_precision(sample), axis=1)))
        print(f"turn vectors in {first.parent}: cosine with fastembed {quantized:.4f}, with full precision {full:.4f}")
        if max(quantized, full) < 0.999:
            print("neither encoder made them; not used")
            continue
        FLAT_ENCODER["name"] = "full" if full > quantized else "fastembed"
        for path in files:
            shutil.copy(path, FLAT / path.name)
        print("turn vectors borrowed; the flat baseline's questions use", FLAT_ENCODER["name"])
        return


def read_turns():
    """Every chat turn: its document, its unit and its words, as the benchmark gave them."""
    turns = []
    for unit_id, doc_id, kind in store["db"].execute("select unit_id, doc_id, kind from unit"):
        if kind == "user" or kind == "assistant":
            start, end = store["unit_bounds"][unit_id]
            text = store["texts"][doc_id][start:end].strip()
            turns.append((doc_id, unit_id, text.split("\n", 1)[-1]))       # without the unit's header line
    return turns


def batched_vectors(texts, name, encode):
    """Vectors for these texts, BATCH at a time, each batch saved under this name."""
    parts = []
    number = 0
    for start in range(0, len(texts), BATCH):
        check_time()
        path = FLAT / f"{name}-{number:03}.npy"
        if path.exists():
            part = np.load(path)
        else:
            part = np.asarray(encode(texts[start:start + BATCH]), dtype=np.float16)
            np.save(path, part)
        parts.append(part.astype(np.float32))
        number += 1
        print(f"{name}: {min(start + BATCH, len(texts))} of {len(texts)}")
    return np.vstack(parts)


def build_turn_index():
    turns = read_turns()
    distinct = {}
    texts = []
    for doc_id, unit_id, text in turns:
        if text not in distinct:
            distinct[text] = len(texts)
            texts.append(text)
    print(len(turns), "turns,", len(texts), "distinct")
    borrow_turns(texts)
    vectors = batched_vectors(texts, "turns", encode_turns)

    of_text = []
    rows = []
    rows_of_document = {}
    for doc_id, unit_id, text in turns:
        of_text.append(distinct[text])
        rows.append((doc_id, unit_id, text))
        if doc_id not in rows_of_document:
            rows_of_document[doc_id] = []
        rows_of_document[doc_id].append(len(rows) - 1)
    return {"vectors": vectors[np.asarray(of_text)], "rows": rows, "rows_of_document": rows_of_document}


def cut_chunks(text):
    """The benchmark's chunking: CHUNK_TOKENS tokens, each start CHUNK_OVERLAP back from the last end."""
    count_tokens("")
    tokens = TOKENIZER.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + CHUNK_TOKENS, len(tokens))
        chunks.append(TOKENIZER.decode(tokens[start:end]))
        if end == len(tokens):
            break
        start += CHUNK_TOKENS - CHUNK_OVERLAP
    return chunks


def build_chunk_index(set_name):
    files = GRAPHRAG_SETS[set_name]
    url = RAW_URL + "/Datasets/Corpus/" + files["corpus"]
    corpus = benchmark_file(files["corpus"], url, files["corpus_sha"])
    if isinstance(corpus, dict):              # the Medical set is one document, the novels a list
        corpus = [corpus]
    rows = []
    rows_of_document = {}
    texts = []
    held = STORES[GRAPHRAG[set_name][0]["store"]]
    for source in corpus:
        path = "graphrag-bench/" + source["corpus_name"] + ".txt"
        for doc_id in path_filter(held, path):
            rows_of_document[doc_id] = []
            number = 1
            for chunk in cut_chunks(source["context"]):
                rows_of_document[doc_id].append(len(rows))
                rows.append((doc_id, number, chunk))
                texts.append(chunk)
                number += 1
    print(len(rows), "chunks of", set_name)
    return {"vectors": batched_vectors(texts, "small-chunks-" + set_name, embed), "rows": rows,
            "rows_of_document": rows_of_document}


def flat_search(index, question_vector, doc_filter):
    """Every row's score, and the rows best first, ties broken by row."""
    scores = index["vectors"] @ question_vector
    if doc_filter is not None:
        keep = np.zeros(len(scores), dtype=bool)
        for doc_id in doc_filter:
            for row in index["rows_of_document"].get(doc_id, []):
                keep[row] = True
        scores = np.where(keep, scores, -np.inf)
    return scores, np.argsort(-scores, kind="stable")

## Block 12: what each arm gives the reader

One function per kind of arm. Each returns the context as a list of strings, its tokens, and the
documents it holds. A turn is dated and named like a record, so the reader sees the same shape from
every arm. A turn longer than the budget left is skipped, as a bundle is.

```
turns:   rows best first; skip a text already shown; add each row that fits the budget
chunks:  rows best first: the top 5 with no budget, else each row that fits the budget
full:    the history's sessions newest first, up to MAX_FULL_TOKENS, put back in date order
store:   retrieve, with the arm's filter, switches and budget
every budget leaves room for PACKET_HEADER, and an arm's tokens count it
none:    nothing
```

In [ ]:
def turn_context(arm, item, doc_filter):
    question_vector = flat_question(item["question"])
    scores, order = flat_search(turn_index(), question_vector, doc_filter)
    rows = order[:FLAT_LOOK].tolist()
    if arm.get("rerank"):
        rows = reranked_rows(item["question"], turn_index(), scores, order)
    context = []
    documents = set()
    seen = set()
    used = 0
    for row in rows:
        if scores[row] == -np.inf:
            break
        doc_id, unit_id, text = turn_index()["rows"][row]
        if text in seen:
            continue
        seen.add(text)
        label, date = unit_label_and_date(store["db"], unit_id)
        line = f"{date} | {document_title(store, doc_id)}: {text}"
        size = count_tokens(line)
        if used + size > arm["budget"]:
            continue
        context.append(line)
        used += size
        documents.add(doc_id)
    return context, used, documents


def chunk_context(arm, item, doc_filter):
    """The chunks most similar to the question, each named by its document and number: TOP_CHUNKS of them when
    the arm has no budget, as the benchmark's plain RAG takes them, else as many as fit the budget."""
    question_vector = embed([QUERY_PREFIX + item["question"]])[0]
    index = chunk_index(item["set"])
    held = STORES[item["store"]]
    scores, order = flat_search(index, question_vector, doc_filter)
    rows = order[:FLAT_LOOK].tolist()
    if arm.get("rerank"):
        rows = reranked_rows(item["question"], index, scores, order)
    context = []
    documents = set()
    used = 0
    for row in rows:
        if scores[row] == -np.inf:
            break
        if arm["budget"] is None and len(context) == TOP_CHUNKS:
            break
        doc_id, number, text = index["rows"][row]
        line = f"{document_title(held, doc_id)}, chunk {number}: {text}"
        size = count_tokens(line)
        if arm["budget"] is not None and used + size > arm["budget"]:
            continue
        context.append(line)
        used += size
        documents.add(doc_id)
    return context, used, documents


def full_context(doc_filter):
    dated = []
    for doc_id in doc_filter:
        dated.append((store["documents"][doc_id]["occurred_at"] or "", doc_id))
    dated.sort(reverse=True)

    context = []
    documents = set()
    used = 0
    for date, doc_id in dated:
        text = f"{date} | {document_title(store, doc_id)}\n{store['texts'][doc_id]}"
        size = count_tokens(text)
        if used + size > MAX_FULL_TOKENS:
            break
        context.append(text)
        used += size
        documents.add(doc_id)
    context.reverse()
    return context, used, documents


def store_context(arm, item, doc_filter, searches):
    context, line = retrieve(STORES[item["store"]], item["question"], doc_filter, keyword=arm["keyword"],
                             vector=arm["vector"], parent=arm["parent"], budget=arm["budget"],
                             hops=arm["hops"], hits=arm["hits"], searches=searches, floor=arm.get("floor"),
                             order=arm.get("order", "score"),
                             items=arm.get("items"), passage=arm.get("passage"), rerank=arm.get("rerank", False),
                             chunks_only=arm.get("chunks_only", False))
    documents = set()
    for record in line["pool"]:
        if record["packed"]:
            documents.add(record["record"][1])
    SOURCES[item["id"]] = sources_of(line)
    parents = 0
    for entry in line["entries"]:
        if entry["record"][0] == "parent":
            parents += 1
    return context, line["tokens"], documents, len(line["entries"]), parents, line["seconds"]


def sources_of(line):
    """The packed records by where they came from: both searches, one search, a parent hit, or the walk."""
    found_by = {}
    for entry in line["entries"]:
        source = "both"
        if entry["rank_w"] is None:
            source = "vector"
        elif entry["rank_v"] is None:
            source = "keyword"
        found_by[tuple(entry["record"])] = source
    counts = {"both": 0, "vector": 0, "keyword": 0, "parent": 0, "walk": 0}
    for record in line["pool"]:
        if not record["packed"]:
            continue
        if record["edge"] == "entry":
            counts[found_by[tuple(record["record"])]] += 1
        elif record["edge"] == "children":
            counts["parent"] += 1
        else:
            counts["walk"] += 1
    return counts


SOURCES = {}


def turn_index():
    if "turns" not in INDEXES:
        INDEXES["turns"] = build_turn_index()
    return INDEXES["turns"]


def chunk_index(set_name):
    if "chunks-" + set_name not in INDEXES:
        INDEXES["chunks-" + set_name] = build_chunk_index(set_name)
    return INDEXES["chunks-" + set_name]


def header_tokens():
    if "header" not in INDEXES:
        INDEXES["header"] = count_tokens(PACKET_HEADER)
    return INDEXES["header"]


def arm_context(arm, item, searches=None):
    """One line for the contexts file. The tokens count the packet's header, and the budget leaves room for it."""
    started = time.time()
    if arm.get("budget") is not None:
        arm = dict(arm)
        arm["budget"] = arm["budget"] - header_tokens()
    doc_filter = None
    if arm["filtered"]:
        doc_filter = path_filter(STORES[item["store"]], item["path"])
    entries = 0
    parents = 0
    if arm["how"] == "store":
        filter_seconds = time.time() - started
        context, tokens, documents, entries, parents, seconds = store_context(arm, item, doc_filter, searches)
        started = time.time() - filter_seconds - seconds     # the shared search counts for every approach
    elif arm["how"] == "turns":
        context, tokens, documents = turn_context(arm, item, doc_filter)
    elif arm["how"] == "chunks":
        context, tokens, documents = chunk_context(arm, item, doc_filter)
    elif arm["how"] == "full":
        context, tokens, documents = full_context(doc_filter)
    else:
        context, tokens, documents = [], 0, set()
    return {"id": item["id"], "context": context, "tokens": tokens + header_tokens(), "seconds": round(time.time() - started, 3),
            "documents": sorted(documents), "entries": entries, "parent_entries": parents,
            "sources": SOURCES.pop(item["id"], None)}


INDEXES = {}

## Block 13: running an arm

First the contexts, one at a time on this one thread. Then the answers, side by side. A call that
fails leaves no line, so a second pass, or the next session, asks that question again. A question
that has a line is never asked twice.

```
run arm:
    contexts = the contexts file, read back
    for each question with no context: make it, write its line, check the time and the spend
    answers = the answers file, read back
    for each question with a context and no answer, 16 at a time:
        answer it, judge it if it is LongMemEval's, write its line
```

In [ ]:
LOCK = threading.Lock()
CALL_POOL = None


def spent_so_far():
    """What earlier sessions spent, from their call log."""
    total = 0.0
    for row in read_jsonl(OUT / "calls.jsonl"):
        total += row.get("cost") or 0.0
    return total


def check_time():
    """Past the time or the spend, a session stops between questions."""
    if time.time() - STARTED > TIME_STOP_HOURS * 3600:
        raise StopRun(f"time stop after {TIME_STOP_HOURS} hours")
    if SPENT >= SPEND_STOP:
        raise StopRun(f"spend stop at ${SPENT:.2f}")


def read_jsonl(path):
    """The rows of a log; a last line cut short by a stopped session is skipped."""
    rows = []
    if not path.exists():
        return rows
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except ValueError:
                pass
    return rows


def read_rows(folder, name):
    rows = {}
    for row in read_jsonl(folder / (name + ".jsonl")):
        rows[row["id"]] = row
    return rows


def append_line(path, row):
    with LOCK:
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def in_parallel(function, jobs, workers):
    """function(*job) for every job, results in the jobs' order."""
    pool = ThreadPoolExecutor(max_workers=workers)
    futures = []
    for job in jobs:
        futures.append(pool.submit(function, *job))
    results = []
    for future in futures:
        results.append(future.result())
    pool.shutdown()
    return results


def answer_one(benchmark, item, context, path, model, concise=False):
    """The reader, then the judge for LongMemEval. A failed call leaves no line, so it is asked again."""
    try:
        response, seconds = ask_reader(item["question"], context, item["asked_on"], model, concise)
        if response is None:
            raise ValueError("the reply held no answer")
        correct = None
        if benchmark == "memory":
            correct = judge(item["check"], item["question"], item["answer"], response)
    except Exception as error:
        print("no answer for", item["id"], repr(error)[:200])
        return
    append_line(path, {"id": item["id"], "response": response, "reader_seconds": seconds, "correct": correct})


def store_contexts(benchmark, arms, items):
    """Each question's context for every store arm still without one, from the same two searches."""
    store_arms = []
    done = {}
    for arm in arms:
        if arm["how"] == "store":
            store_arms.append(arm)
            done[arm["name"]] = set(read_rows(CONTEXTS, benchmark + "-" + arm["name"]))
    waiting = []
    for item in items:
        for arm in store_arms:
            if item["id"] not in done[arm["name"]]:
                waiting.append(item)
                break
    for start in range(0, len(waiting), SEARCH_BATCH):
        check_time()
        batch = waiting[start:start + SEARCH_BATCH]
        prefetch_keywords(store_arms, batch)
        for item in batch:
            searches = {}
            for arm in store_arms:
                if item["id"] in done[arm["name"]]:
                    continue
                append_line(CONTEXTS / (benchmark + "-" + arm["name"] + ".jsonl"), arm_context(arm, item, searches))
        KEYWORDS.clear()
    print(benchmark, "store contexts done,", round((time.time() - STARTED) / 3600, 2), "hours in")


def run_arm(benchmark, arm, items):
    """All of one arm's contexts, then all of its answers."""
    name = benchmark + "-" + arm["name"]
    if arm["how"] == "turns":                 # built before the clock on any question starts
        turn_index()
    if arm["how"] == "chunks":
        chunk_index(benchmark)
    context_path = CONTEXTS / (name + ".jsonl")
    answer_path = ANSWERS / (name + ".jsonl")

    contexts = {}
    for row in read_jsonl(context_path):
        contexts[row["id"]] = row
    for item in items:
        if item["id"] not in contexts:
            check_time()
            row = arm_context(arm, item)
            append_line(context_path, row)
            contexts[item["id"]] = row
    print(name, "contexts done,", round((time.time() - STARTED) / 3600, 2), "hours in")

    answered = set()
    for row in read_jsonl(answer_path):
        answered.add(row["id"])
    jobs = []
    for item in items:
        if item["id"] not in answered:
            jobs.append((benchmark, item, contexts[item["id"]]["context"], answer_path, arm["reader"],
                         arm.get("concise", False)))

    workers = WORKERS
    if arm["how"] == "full":
        workers = FULL_WORKERS
    for start in range(0, len(jobs), 64):
        check_time()
        in_parallel(answer_one, jobs[start:start + 64], workers)
    print(f"{name} answers done, ${SPENT:.2f} so far")


SPENT = spent_so_far()
FINISHED = False


if not KEY:
    raise SystemExit("no OPENAI_API_KEY: start this notebook from the editor with the secret attached")

def all_answered():
    """True when every arm has an answer to every question."""
    sets = [("memory", MEMORY_ARMS, MEMORY)]
    for set_name in GRAPHRAG:
        sets.append((set_name, arms_for(set_name), GRAPHRAG[set_name]))
    for benchmark, arms, items in sets:
        for arm in arms:
            answers = read_rows(ANSWERS, benchmark + "-" + arm["name"])
            for item in items:
                if item["id"] not in answers:
                    return False
    return True


with open(STORE_DIR / "threadatlas.sqlite", "rb") as f:      # the first arm starts from the file in memory, as the others do
    while f.read(1 << 24):
        pass

try:
    for attempt in range(2):                  # the second pass asks what a dropped call left unanswered
        if len(MEMORY) > 0:
            store_contexts("memory", MEMORY_ARMS, MEMORY)
            for arm in MEMORY_ARMS:
                run_arm("memory", arm, MEMORY)
        for set_name in GRAPHRAG:
            store_contexts(set_name, arms_for(set_name), GRAPHRAG[set_name])
            for arm in arms_for(set_name):
                run_arm(set_name, arm, GRAPHRAG[set_name])
    FINISHED = all_answered()
except StopRun as stop:
    print(f"{stop}. Attach this output and run again to go on.")

## Block 14: GraphRAG-Bench's evaluator

This block runs the benchmark's own code from one pinned commit, unchanged. `Evaluation.generation_eval`
gives answer correctness, ROUGE-L, coverage and faithfulness. `Evaluation.retrieval_eval` gives
context relevance and evidence recall. Both call `gpt-4o-mini`, and answer correctness uses
`bge-large-en-v1.5`, as the benchmark's README sets them. The evaluator's calls are not in
`calls.jsonl`. The PyPI package `Evaluation` that its requirements file names is not installed.

It runs in its own environment. An arm already scored is skipped.

```
download the repository at the pinned commit, and install what it imports into a new environment
generation_eval for every arm with all its answers, then retrieval_eval for every quality arm:
    arms in their order, each on every GraphRAG-Bench set before the next arm
    write the arm as the evaluator's input file
    SCORERS (8) at once, each in its own process, each with its own log
```

In [ ]:
SCORER = Path("/tmp/graphrag-benchmark")
SCORER_PACKAGES = ["datasets==3.3.2", "langchain==0.3.26", "langchain_core==0.3.69", "langchain_openai==0.3.28",
                   "langchain_community==0.3.27", "langchain_ollama==0.3.5", "numpy", "pydantic==2.11.7", "ragas==0.2.15",
                   "rouge_score==0.1.2", "json5", "json_repair", "sentence-transformers==6.2.0"]


def scorer_python():
    return SCORER / "env" / "bin" / "python"


def set_up_scorer():
    """The benchmark's repository at its commit, in an environment of its own."""
    if scorer_python().exists():
        return
    SCORER.mkdir(parents=True, exist_ok=True)
    archive = SCORER / "repository.tar.gz"
    urllib.request.urlretrieve(SCORER_URL, archive)
    with tarfile.open(archive) as tar:
        tar.extractall(SCORER)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)    # Kaggle's Python has no ensurepip for venv
    subprocess.run([sys.executable, "-m", "virtualenv", str(SCORER / "env")], check=True)
    subprocess.run([str(scorer_python()), "-m", "pip", "install", "-q"] + SCORER_PACKAGES, check=True)


def write_scorer_input(set_name, arm_name):
    """One arm in the evaluator's format, or nothing if the arm is not finished."""
    name = set_name + "-" + arm_name
    contexts = {}
    for row in read_jsonl(CONTEXTS / (name + ".jsonl")):
        contexts[row["id"]] = row["context"]
    answers = {}
    for row in read_jsonl(ANSWERS / (name + ".jsonl")):
        answers[row["id"]] = row["response"]

    rows = []
    for item in GRAPHRAG[set_name]:
        if item["id"] not in answers:
            return None
        rows.append({"id": item["id"], "question": item["question"], "question_type": item["type"],
                     "source": item["source"], "ground_truth": item["answer"], "evidence": item["evidence"],
                     "generated_answer": str(answers[item["id"]]), "context": contexts[item["id"]]})
    path = SCORES / (name + "-input.json")
    path.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
    return path


def start_scorer(module, data_file, output_file, log_file):
    environment = dict(os.environ)
    environment["LLM_API_KEY"] = KEY
    folder = SCORER / ("GraphRAG-Benchmark-" + SCORER_COMMIT)
    command = [str(scorer_python()), "-m", module, "--mode", "API", "--model", SCORER_MODEL,
               "--base_url", "https://api.openai.com/v1", "--embedding_model", LARGE_MODEL,
               "--data_file", str(data_file), "--output_file", str(output_file), "--detailed_output"]
    log = open(log_file, "w", encoding="utf-8")
    return subprocess.Popen(command, cwd=folder, env=environment, stdout=log, stderr=subprocess.STDOUT), output_file, log_file


def wait_for(running):
    global FINISHED
    for process, output_file, log_file in running:
        process.wait()
        if not output_file.exists():
            FINISHED = False
            print("the evaluator failed; the end of", log_file.name, "reads:")
            print(log_file.read_text(encoding="utf-8", errors="replace")[-1500:])


def score_graphrag():
    """Answer correctness for every arm before retrieval quality, each arm on every set before the next arm."""
    set_up_scorer()
    running = []
    for module, kind in (("Evaluation.generation_eval", "generation"), ("Evaluation.retrieval_eval", "retrieval")):
        for set_name in GRAPHRAG:
            for arm in arms_for(set_name):
                name = set_name + "-" + arm["name"]
                output_file = SCORES / (name + "-" + kind + ".json")
                if output_file.exists() or (kind == "retrieval" and not arm["quality"]):
                    continue
                data_file = write_scorer_input(set_name, arm["name"])
                if data_file is None:
                    continue
                log_file = SCORES / (name + "-" + kind + ".log")
                if len(running) == 0:                 # never start a batch that could run into Kaggle's limit
                    check_time()
                running.append(start_scorer(module, data_file, output_file, log_file))
                if len(running) == SCORERS:
                    wait_for(running)
                    running = []
    wait_for(running)


if FINISHED and len(GRAPHRAG) > 0:
    try:
        score_graphrag()
    except StopRun as stop:
        FINISHED = False
        print(f"{stop}. Attach this output and run again to score the rest.")
    except (OSError, subprocess.CalledProcessError) as error:
        FINISHED = False
        print("the evaluator did not run, and the answers are saved:", repr(error)[:300])

## Block 14b: each benchmark's best setting, and its own arms

Once every arm of the grid has its scores, each benchmark's best setting is the grid's store arm with
the benchmark's highest figure: LongMemEval's accuracy, or GraphRAG-Bench's mean over the four types
of their mean answer correctness, on the questions every scored arm of the set kept. The best
setting then runs widened, with the vector search alone, with parents off, in date order, with the
reader asked for a short answer, with the floor, and reranked by the cross-encoder, beside plain RAG
(flat retrieval on LongMemEval) reranked the same way at the same budget; the best setting at 2,000 tokens runs with the
floor too. On LongMemEval the best setting also runs with no filter, read by gpt-4o, and at
OmniMemEval's 20 items on our reader and on gpt-4.1-mini. Those arms are answered and scored like the rest.

```
common(set) = the questions whose answer correctness every scored arm of the set has, no context aside
best(benchmark) = the grid arm with the highest figure; ties go to the arm listed first
its arms: wide, vector only, parents off, date order, concise answers, the floor (also on the best at 2,000),
    reranked, and the baseline reranked at the same budget;
    LongMemEval: also no filter, gpt-4o, and 20 items on our reader and on gpt-4.1-mini
```

In [ ]:
def share(part, whole):
    if whole == 0:
        return None
    return round(100 * part / whole, 2)


def mean_of(values):
    kept = []
    for value in values:
        if value is not None and value == value:           # a NaN is the evaluator's "could not score"
            kept.append(value)
    if len(kept) == 0:
        return None
    return round(100 * sum(kept) / len(kept), 2)


def read_scores(name):
    """question id -> {metric: score} from the evaluator's two output files for one set's arm."""
    scores = {}
    for kind in ("generation", "retrieval"):
        path = SCORES / (name + "-" + kind + ".json")
        if not path.exists():
            continue
        results = json.loads(path.read_text(encoding="utf-8"))
        for question_type in results:
            for row in results[question_type]["detailed"]:
                if row["id"] not in scores:
                    scores[row["id"]] = {"type": question_type}
                for metric in row["metrics"]:
                    scores[row["id"]][metric] = row["metrics"][metric]
    return scores


def common_questions(set_name):
    """The questions every scored arm of a set has an answer correctness for; no context is left out."""
    common = None
    for arm in arms_for(set_name):
        if arm["how"] == "none":
            continue
        scores = read_scores(set_name + "-" + arm["name"])
        if len(scores) == 0:
            continue
        kept = set()
        for question_id in scores:
            value = scores[question_id].get("answer_correctness")
            if value is not None and value == value:
                kept.add(question_id)
        if common is None:
            common = kept
        else:
            common = common & kept
    if common is None:
        return set()
    return common


def figure_of(benchmark, arm_name, common):
    """The benchmark's own figure for one arm, or None when it has no answers or scores yet."""
    if benchmark == "memory":
        answers = read_rows(ANSWERS, "memory-" + arm_name)
        if len(answers) < len(MEMORY):
            return None
        right = 0
        for item in MEMORY:
            if answers[item["id"]]["correct"]:
                right += 1
        return share(right, len(MEMORY))
    scores = read_scores(benchmark + "-" + arm_name)
    by_type = {}
    for question_id in common:
        if question_id not in scores:
            return None
        question_type = scores[question_id]["type"]
        if question_type not in by_type:
            by_type[question_type] = []
        by_type[question_type].append(scores[question_id]["answer_correctness"])
    if len(by_type) == 0:
        return None
    means = []
    for question_type in by_type:
        means.append(sum(by_type[question_type]) / len(by_type[question_type]))
    return round(100 * sum(means) / len(means), 2)


def grid_arms(benchmark):
    arms = MEMORY_ARMS
    if benchmark != "memory":
        arms = arms_for(benchmark)
    found = []
    for arm in arms:
        if arm["how"] == "store" and arm["reader"] == READER and arm["keyword"] and arm["vector"] and arm["parent"]:
            if arm not in EXTRAS.get(benchmark, []):
                found.append(arm)
    return found


def best_setting(benchmark, budget=None):
    """The grid arm with the benchmark's highest figure, at one budget when one is named, or None until every
    grid arm has a figure."""
    common = set()
    if benchmark != "memory":
        common = common_questions(benchmark)
    best = None
    best_figure = None
    for arm in grid_arms(benchmark):
        if budget is not None and arm["budget"] != budget:
            continue
        figure = figure_of(benchmark, arm["name"], common)
        if figure is None:
            return None
        if best is None or figure > best_figure:
            best = arm
            best_figure = figure
    return best


def varied(best, suffix, **changes):
    arm = dict(best)
    arm["name"] = best["name"] + suffix
    arm["quality"] = False
    for key in changes:
        arm[key] = changes[key]
    return arm


def best_arms(benchmark, best, best_small):
    """The best setting widened, with the vector search alone, with parents off, in date order, with short answers,
    and with the floor; the floor also on the best setting at the smallest budget. On LongMemEval: no filter,
    gpt-4o, and OmniMemEval's 20 items on our reader and on its own."""
    arms = []
    passage = WIDE_PASSAGE
    if benchmark == "memory":
        passage = None                                  # a chat turn is shown whole already
    arms.append(varied(best, "-wide", hits=WIDE_HITS, passage=passage))
    arms.append(varied(best, "-vector-only", keyword=False))
    arms.append(varied(best, "-parent-off", parent=False))
    arms.append(varied(best, "-date-order", order="date"))
    arms.append(varied(best, "-concise", concise=True))
    arms.append(varied(best, "-floor", floor=True))
    arms.append(varied(best, "-rerank", rerank=True))
    if benchmark == "memory":
        arms.append(other_arm("flat-" + str(best["budget"]) + "-rerank", "turns", filtered=best["filtered"],
                              budget=best["budget"], rerank=True))
    elif best["filtered"]:
        arms.append(other_arm("plain-rag-" + str(best["budget"]) + "-rerank", "chunks", budget=best["budget"], rerank=True))
    else:
        arms.append(other_arm("plain-rag-no-filter-" + str(best["budget"]) + "-rerank", "chunks", filtered=False,
                              budget=best["budget"], rerank=True))
    if best_small is not None and best_small["name"] != best["name"]:
        arms.append(varied(best_small, "-floor", floor=True))
    if benchmark == "memory":
        if best["filtered"]:
            arms.append(varied(best, "", filtered=False, name="no-filter-" + best["name"]))
        arms.append(varied(best, "-gpt-4o", reader=STRONGER_READER))
        arms.append(varied(best, "-omni" + str(OMNI_ITEMS + 1), items=OMNI_ITEMS, budget=100000))
        arms.append(varied(best, "-omni" + str(OMNI_ITEMS + 1) + "-" + OMNI_READER, items=OMNI_ITEMS, budget=100000,
                           reader=OMNI_READER))
    return arms


BEST = {}
if FINISHED and BEST_ARMS:
    try:
        benchmarks = []
        if len(MEMORY) > 0:
            benchmarks.append("memory")
        for set_name in GRAPHRAG:
            benchmarks.append(set_name)
        for benchmark in benchmarks:
            best = best_setting(benchmark)
            if best is None:
                FINISHED = False
                print(benchmark, "has no best setting yet: the grid is not all scored")
                continue
            best_small = best_setting(benchmark, BUDGETS[0])
            BEST[benchmark] = best["name"]
            print(benchmark, "best setting:", best["name"], "; best at", BUDGETS[0], "tokens:", best_small["name"])
            EXTRAS[benchmark] = best_arms(benchmark, best, best_small)
            items = MEMORY
            if benchmark == "memory":
                MEMORY_ARMS = MEMORY_ARMS + EXTRAS[benchmark]
            else:
                items = GRAPHRAG[benchmark]
            store_contexts(benchmark, EXTRAS[benchmark], items)
            for arm in EXTRAS[benchmark]:
                run_arm(benchmark, arm, items)
        if FINISHED:
            FINISHED = all_answered()
        if FINISHED and len(GRAPHRAG) > 0:
            score_graphrag()
    except StopRun as stop:
        FINISHED = False
        print(f"{stop}. Attach this output and run again to go on.")

## Block 15: the tables

LongMemEval first, by the benchmark's own three figures: overall accuracy, accuracy averaged over
the six types, and accuracy on the unanswerable questions, then one line per type, then the context
and latency Zep's Table 2 reports. A store arm is compared with flat retrieval at the same budget
and reader, question by question, by McNemar's exact test on the questions one got and the other
missed. `reached` is
the share of answerable questions whose marked turns all stood in the context. The seconds are
retrieval alone. `parent entries` is the share of entries that were parents.

Each GraphRAG-Bench set has a table with the columns of the paper's Tables 3 and 4 and that set's
published plain RAG row beside it, over the questions every scored arm kept. A store arm is
compared with plain RAG at the same budget and filter, question by question. These tables print
what exists.

```
for each memory arm with answers:
    accuracy overall, by type, averaged over types, on abstention, with its standard error
    tokens, retrieval seconds (median, IQR), reached, misses by cause
    against flat retrieval at the same budget and reader: won, lost, exact p
for each GraphRAG-Bench set, for each arm with scores:
    mean of each of the evaluator's metrics, by question type, as percentages
    tokens, against plain RAG at the same budget and filter on answer correctness
```

In [ ]:
PUBLISHED_MEMORY = {"Zep, gpt-4o-mini": 63.8, "full context, gpt-4o-mini (Zep's paper)": 55.4,
                    "Zep, gpt-4o": 71.2, "full context, gpt-4o (Zep's paper)": 60.2}

# the leaderboard's "RAG (w/o rerank)" row for each set; the novels' evidence recall and context
# relevancy are from the paper's Table 4, which the leaderboard does not show for Medical
PUBLISHED_PLAIN_RAG = {
    "novels": {"Fact Retrieval": {"answer_correctness": 58.76, "rouge_score": 37.35, "evidence_recall": 61.37, "context_relevancy": 74.66},
               "Complex Reasoning": {"answer_correctness": 41.35, "rouge_score": 15.12, "evidence_recall": 59.80, "context_relevancy": 80.82},
               "Contextual Summarize": {"answer_correctness": 50.08, "coverage_score": 82.53, "evidence_recall": 69.08, "context_relevancy": 80.05},
               "Creative Generation": {"answer_correctness": 41.52, "faithfulness": 47.46, "coverage_score": 37.84, "evidence_recall": 32.48, "context_relevancy": 82.84}},
    "medical": {"Fact Retrieval": {"answer_correctness": 63.72, "rouge_score": 29.21},
                "Complex Reasoning": {"answer_correctness": 57.61, "rouge_score": 13.98},
                "Contextual Summarize": {"answer_correctness": 63.72, "coverage_score": 77.34},
                "Creative Generation": {"answer_correctness": 58.94, "faithfulness": 35.88, "coverage_score": 57.87}},
}

MEMORY_TYPES = ("single-session-user", "single-session-assistant", "single-session-preference", "multi-session",
                "temporal-reasoning", "knowledge-update")


def middle_and_spread(values):
    values = sorted(values)
    n = len(values)
    if n % 2 == 1:
        median = values[n // 2]
    else:
        median = (values[n // 2 - 1] + values[n // 2]) / 2
    quarter = values[n // 4]
    three_quarters = values[(3 * n) // 4]
    return median, three_quarters - quarter


def exact_p(won, lost):
    """McNemar's exact test: the two-sided binomial tail of the questions the arms split on."""
    n = won + lost
    if n == 0:
        return 1.0
    tail = 0
    for k in range(min(won, lost) + 1):
        tail += math.comb(n, k)
    return min(1.0, 2 * tail / 2 ** n)


def memory_line(arm_name, base):
    """Every number for one LongMemEval arm."""
    name = "memory-" + arm_name
    contexts = read_rows(CONTEXTS, name)
    answers = read_rows(ANSWERS, name)
    if len(answers) == 0:
        return None

    right = 0
    total = 0
    by_type = {}
    abstention = [0, 0]
    tokens = []
    seconds = []
    entries = 0
    parents = 0
    reached = 0
    answerable = 0
    lost_by_retrieval = 0
    lost_by_reader = 0
    won = 0
    lost = 0
    for item in MEMORY:
        row = answers.get(item["id"])
        if row is None:
            continue
        context = contexts[item["id"]]
        total += 1
        correct = 0
        if row["correct"]:
            correct = 1
        right += correct
        if item["type"] not in by_type:
            by_type[item["type"]] = [0, 0]
        by_type[item["type"]][0] += correct
        by_type[item["type"]][1] += 1
        if item["check"] == "abstention":
            abstention[0] += correct
            abstention[1] += 1
        tokens.append(context["tokens"])
        seconds.append(context["seconds"])
        entries += context["entries"]
        parents += context["parent_entries"]

        if item["check"] != "abstention" and len(item["marked_turns"]) > 0:
            answerable += 1
            text = " ".join(" ".join(context["context"]).split())
            whole = True
            for turn in item["marked_turns"]:
                if not shows_turn(text, turn):
                    whole = False
            if whole:
                reached += 1
            if not correct and whole:
                lost_by_reader += 1
            if not correct and not whole:
                lost_by_retrieval += 1

        if base and item["id"] in base:
            if row["correct"] and not base[item["id"]]["correct"]:
                won += 1
            if base[item["id"]]["correct"] and not row["correct"]:
                lost += 1

    types = {}
    averages = []
    for question_type in by_type:
        types[question_type] = share(by_type[question_type][0], by_type[question_type][1])
        averages.append(by_type[question_type][0] / by_type[question_type][1])
    accuracy = right / total
    median, spread = middle_and_spread(seconds)
    return {"arm": arm_name, "questions": total, "correct": right, "accuracy": share(right, total),
            "standard_error": round(100 * math.sqrt(accuracy * (1 - accuracy) / total), 2),
            "task_averaged": round(100 * sum(averages) / len(averages), 2),
            "abstention": share(abstention[0], abstention[1]), "by_type": types,
            "mean_tokens": round(sum(tokens) / len(tokens)), "median_seconds": round(median, 2),
            "seconds_iqr": round(spread, 2), "reached": share(reached, answerable),
            "lost_by_retrieval": lost_by_retrieval, "lost_by_reader": lost_by_reader,
            "parent_entries": share(parents, entries), "won": won, "lost": lost, "p": round(exact_p(won, lost), 4)}


def flat_of(arm):
    """The baseline an arm is compared with: raw retrieval at the arm's budget and reader, filtered as the arm is."""
    if arm["how"] != "store" or arm["reader"] != READER:
        return None
    name = "flat-no-filter-" + str(arm["budget"])
    if arm["filtered"]:
        name = "flat-" + str(arm["budget"])
    if arm.get("rerank"):
        name += "-rerank"
    return name


def memory_table():
    lines = []
    for arm in MEMORY_ARMS:
        base = None
        if flat_of(arm) is not None:
            base = read_rows(ANSWERS, "memory-" + flat_of(arm))
        line = memory_line(arm["name"], base)
        if line is not None:
            lines.append(line)

    print(f"{'LongMemEval arm':32} {'n':>4} {'acc':>6} {'se':>5} {'task':>6} {'abs':>6} {'tokens':>7} {'sec':>6} {'iqr':>6}"
          f" {'reach':>6} {'ret':>4} {'read':>4} {'par%':>5} {'won':>4} {'lost':>4} {'p':>7}")
    for line in lines:
        print(f"{line['arm']:32} {line['questions']:>4} {line['accuracy']:>6} {line['standard_error']:>5}"
              f" {line['task_averaged']:>6} {str(line['abstention']):>6} {line['mean_tokens']:>7}"
              f" {line['median_seconds']:>6} {line['seconds_iqr']:>6} {str(line['reached']):>6}"
              f" {line['lost_by_retrieval']:>4} {line['lost_by_reader']:>4} {str(line['parent_entries']):>5}"
              f" {line['won']:>4} {line['lost']:>4} {line['p']:>7}")
    print("won, lost and p are against flat retrieval at the same budget and reader")
    print()
    print(f"{'by type':32}", end="")
    for question_type in MEMORY_TYPES:
        print(f" {question_type[:14]:>14}", end="")
    print()
    for line in lines:
        print(f"{line['arm']:32}", end="")
        for question_type in MEMORY_TYPES:
            print(f" {str(line['by_type'].get(question_type)):>14}", end="")
        print()
    print()
    for name in PUBLISHED_MEMORY:
        print("published,", name, PUBLISHED_MEMORY[name])
    return lines


def novel_line(set_name, arm, base, common):
    """Every number for one GraphRAG-Bench arm on one set, on the questions every scored arm kept."""
    scores = read_scores(set_name + "-" + arm["name"])
    if len(scores) == 0:
        return None
    kept = common
    if arm["how"] == "none":
        kept = set(scores)
    metrics = {}
    for question_id in scores:
        if question_id not in kept:
            continue
        row = scores[question_id]
        if row["type"] not in metrics:
            metrics[row["type"]] = {}
        for metric in row:
            if metric != "type":
                if metric not in metrics[row["type"]]:
                    metrics[row["type"]][metric] = []
                metrics[row["type"]][metric].append(row[metric])
    table = {}
    means = []
    for question_type in metrics:
        table[question_type] = {}
        for metric in metrics[question_type]:
            table[question_type][metric] = mean_of(metrics[question_type][metric])
        if table[question_type].get("answer_correctness") is not None:
            means.append(table[question_type]["answer_correctness"])
    figure = None
    if len(means) > 0:
        figure = round(sum(means) / len(means), 2)

    differences = []
    if base:
        for question_id in common:
            if question_id in scores and question_id in base:
                differences.append(scores[question_id]["answer_correctness"] - base[question_id]["answer_correctness"])
    difference = None
    error = None
    if len(differences) > 1:
        average = sum(differences) / len(differences)
        spread = 0.0
        for value in differences:
            spread += (value - average) ** 2
        difference = round(100 * average, 2)
        error = round(100 * math.sqrt(spread / (len(differences) - 1) / len(differences)), 2)

    tokens = []
    entries = 0
    parents = 0
    sources = {}
    for row in read_jsonl(CONTEXTS / (set_name + "-" + arm["name"] + ".jsonl")):
        tokens.append(row["tokens"])
        entries += row["entries"]
        parents += row["parent_entries"]
        for source in row.get("sources") or {}:
            sources[source] = sources.get(source, 0) + row["sources"][source]
    scored = 0
    for question_id in kept:
        if question_id in scores:
            scored += 1
    return {"arm": arm["name"], "questions": len(scores), "scored": scored, "paired": len(differences), "by_type": table,
            "figure": figure, "mean_tokens": round(sum(tokens) / len(tokens)), "against_plain_rag": difference,
            "standard_error": error, "parent_entries": share(parents, entries), "sources": sources}


def plain_of(arm):
    """The baseline an arm is compared with: plain RAG at the arm's budget, filtered as the arm is."""
    if arm["how"] != "store":
        return None
    name = "plain-rag-no-filter-" + str(arm["budget"])
    if arm["filtered"]:
        name = "plain-rag-" + str(arm["budget"])
    if arm.get("rerank"):
        name += "-rerank"
    return name


def novel_table(set_name):
    print("GraphRAG-Bench,", set_name)
    published = PUBLISHED_PLAIN_RAG[set_name]
    common = common_questions(set_name)
    lines = []
    for arm in arms_for(set_name):
        base = None
        if plain_of(arm) is not None:
            base = read_scores(set_name + "-" + plain_of(arm))
        line = novel_line(set_name, arm, base, common)
        if line is not None:
            lines.append(line)
    columns = ("answer_correctness", "rouge_score", "coverage_score", "faithfulness", "evidence_recall", "context_relevancy")
    for question_type in published:
        print(question_type)
        print(f"  {'arm':30}", end="")
        for column in columns:
            print(f" {column[:10]:>10}", end="")
        print()
        for line in lines:
            print(f"  {line['arm']:30}", end="")
            for column in columns:
                print(f" {str(line['by_type'].get(question_type, {}).get(column)):>10}", end="")
            print()
        print(f"  {'published plain RAG':30}", end="")
        for column in columns:
            print(f" {str(published[question_type].get(column)):>10}", end="")
        print()
    print()
    for line in lines:
        print(f"{line['arm']:30} mean of types {str(line['figure']):>6}   scored {line['scored']} of {line['questions']}"
              f"   tokens {line['mean_tokens']:>6}   against plain RAG {line['against_plain_rag']} +- {line['standard_error']}"
              f" on {line['paired']}   parent entries {line['parent_entries']}%   packed by source {line['sources']}")
    print(len(common), "questions kept by every scored arm; no context is scored on its own")
    print("published mean tokens: plain RAG 879, HippoRAG2 1,008, RAPTOR 3,441, Fast-GraphRAG 4,204, LightRAG 100,832")
    return lines


MEMORY_LINES = []
if len(MEMORY) > 0:
    MEMORY_LINES = memory_table()
    print()
GRAPHRAG_LINES = {}
for set_name in GRAPHRAG:
    GRAPHRAG_LINES[set_name] = novel_table(set_name)
    print()

## Block 15b: the grid

Each benchmark's grid of hops by budget, filtered and unfiltered, by the benchmark's own figure, with
its best setting marked.

In [ ]:
def grid_table(benchmark):
    common = set()
    if benchmark != "memory":
        common = common_questions(benchmark)
    rows = {}
    print("grid,", benchmark, "(* best)")
    print(f"  {'scope':10} {'hops':>4}", end="")
    for budget in BUDGETS:
        print(f" {budget:>8}", end="")
    print()
    for prefix, scope in (("", "filtered"), ("no-filter-", "no filter")):
        for hops in (0, 1, 2):
            names = []
            for budget in BUDGETS:
                names.append(prefix + "hop" + str(hops) + "-" + str(budget))
            figures = []
            for name in names:
                figures.append(figure_of(benchmark, name, common))
                rows[name] = figures[-1]
            if figures.count(None) == len(figures):
                continue
            print(f"  {scope:10} {hops:>4}", end="")
            for i in range(len(names)):
                mark = " "
                if BEST.get(benchmark) == names[i]:
                    mark = "*"
                print(f" {str(figures[i]):>7}{mark}", end="")
            print()
    print()
    return {"best": BEST.get(benchmark), "figures": rows}


APPROACHES = {}
if len(MEMORY) > 0:
    APPROACHES["memory"] = grid_table("memory")
for set_name in GRAPHRAG:
    APPROACHES[set_name] = grid_table(set_name)

## Block 16: bodies of work against the baseline

The model's groups were scored at the end of the global-layer notebook. Here they stand beside a
baseline: average linkage on the abstracts' vectors, cut at cosine distance .30, smallest body 5.
The baseline's rows are marked tuned. Both are scored the same way: source folders as classes, a novel its own class,
on all the works and on the held-out works of dumps 6 to 10.

```
works = every document that is not a chat, its class its source folder, a novel its own
baseline = average linkage over the abstracts' vectors, cut at distance .30
           a body under 5 works: each of its works stands alone
model = the groups in library-groups.json
score each on all works and on dumps 6 to 10: harmonic purity, ARI, NMI, novels left alone
```

In [ ]:
DUMPS = 10


def year_of(date):
    """The year a date opens with, which may be negative. An undated document sorts last."""
    date = date or ""
    sign = 1
    if date.startswith("-"):
        sign = -1
        date = date[1:]
    digits = ""
    for c in date:
        if not c.isdigit():
            break
        digits += c
    if not digits:
        return 10 ** 6
    return sign * int(digits)


def by_date(doc_ids, docs):
    """Documents sorted by year, date as written, path."""
    rows = []
    for d in doc_ids:
        rows.append((year_of(docs[d]["occurred_at"]), docs[d]["occurred_at"] or "", docs[d]["source_uri"], d))
    rows.sort()
    ordered = []
    for row in rows:
        ordered.append(row[3])
    return ordered


def source_set(doc):
    """The folder a document came from."""
    return PurePosixPath(doc["source_uri"]).parent.as_posix()


def read_dumps(docs):
    """The corpus as DUMPS lists of document ids, each a date-ordered run of every source folder."""
    sets = {}
    for doc_id, doc in docs.items():
        sets.setdefault(source_set(doc), []).append(doc_id)
    dumps = []
    for number in range(DUMPS):
        dumps.append([])
    for members in sets.values():
        for position, doc_id in enumerate(by_date(members, docs)):
            dumps[position * DUMPS // len(members)].append(doc_id)
    ordered = []
    for dump in dumps:
        ordered.append(by_date(dump, docs))
    return ordered


def held_out_works(works):
    """The works in dumps 6 to 10, cut exactly as the global layer cut them."""
    inside = set(works)
    held_out = []
    for dump in read_dumps(store["documents"])[5:]:
        for doc_id in dump:
            if doc_id in inside:
                held_out.append(doc_id)
    return held_out


def purity(found, truth):
    """The share of items in the largest true class of their found group."""
    counts = {}
    for f, t in zip(found, truth):
        counts.setdefault(f, {})
        counts[f][t] = counts[f].get(t, 0) + 1
    right = 0
    for classes in counts.values():
        right += max(classes.values())
    return right / len(found)


def score_labels(labels, works):
    """A grouping against the source folders."""
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    size = {}
    for doc_id in labels:
        size[labels[doc_id]] = size.get(labels[doc_id], 0) + 1
    truth = []
    found = []
    novels = 0
    alone = 0
    for doc_id in works:
        found.append(labels[doc_id])
        document = store["documents"][doc_id]
        if "graphrag-bench" in document["source_uri"]:
            truth.append(doc_id)
            novels += 1
            if size[labels[doc_id]] == 1:
                alone += 1
        else:
            truth.append(source_set(document))
    forward = purity(found, truth)
    backward = purity(truth, found)
    return {"works": len(works), "harmonic_purity": round(2 * forward * backward / (forward + backward), 3),
            "ari": round(adjusted_rand_score(truth, found), 3), "nmi": round(normalized_mutual_info_score(truth, found), 3),
            "novels_left_alone": alone, "novels": novels}


def baseline_labels(works):
    """Average linkage on the abstracts, cut at cosine distance .30; a body under 5 falls apart."""
    from scipy.cluster.hierarchy import fcluster, linkage
    texts = []
    for doc_id in works:
        found = abstract_above(store, doc_id, store["document_node"].get(doc_id))
        text = ""
        if found:
            text = store["db"].execute("select text from abstract where doc_id = ? and node_id = ?",
                                       (doc_id, found[0][2])).fetchone()[0]
        texts.append(text)
    clusters = fcluster(linkage(embed(texts), "average", "cosine"), 0.30, "distance").tolist()
    size = {}
    for number in clusters:
        size[number] = size.get(number, 0) + 1
    labels = {}
    for doc_id, number in zip(works, clusters):
        if size[number] >= 5:
            labels[doc_id] = "body " + str(number)
        else:
            labels[doc_id] = doc_id
    return labels


def model_labels():
    state = json.loads((STORE_DIR / "library-groups.json").read_text(encoding="utf-8"))
    labels = {}
    for group in state["groups"]:
        for doc_id in group["works"]:
            labels[doc_id] = group["number"]
    return labels


def bodies_table():
    works = []
    for doc_id in sorted(store["documents"]):
        if "chats/" not in store["documents"][doc_id]["source_uri"]:
            works.append(doc_id)
    if len(works) < 2:
        print(len(works), "work in this store: no bodies of work to score")
        return None
    if not (STORE_DIR / "library-groups.json").exists():
        print("library-groups.json is not attached with this store: no bodies of work to score")
        return None
    held_out = held_out_works(works)
    model = model_labels()
    baseline = baseline_labels(works)
    table = {"model, all": score_labels(model, works), "model, dumps 6 to 10": score_labels(model, held_out),
             "baseline .30 (tuned), all": score_labels(baseline, works),
             "baseline .30 (tuned), dumps 6 to 10": score_labels(baseline, held_out)}
    for name in table:
        print(f"{name:38}", table[name])
    return table


BODIES = bodies_table()

## Block 17: five questions read by eye

Five questions from the project's own corpus. Each searches its own source folder through the same
path, and the context and answer go to `traced.txt`. They are not scored.

- **Tip becomes Ozma.** One character, two names, a change at the end of Oz book 2.
- **Watson's wound.** Doyle puts it in the shoulder in one book and the leg in another.
- **The deerstalker.** Doyle never wrote it.
- **Helen at Troy.** Homer and Euripides disagree.
- **Dorothy in book 2.** The traced query for Dr. Fang. She is not in that book.

```
for each case: context = retrieve(question, the folder's documents)
               answer = reader(question, context)
               write the question, the answer and the whole context to traced.txt
```

In [ ]:
TRACED = [("How does Tip become Ozma, and who turned Ozma into a boy?", "/oz/"),
          ("Where was Dr. Watson wounded?", "/holmes/"),
          ("Does Sherlock Holmes wear a deerstalker?", "/holmes/"),
          ("Was Helen at Troy during the war?", "/greek/"),
          ("What does Dorothy do in the second Oz book, The Marvelous Land of Oz?", "/oz/")]


def trace_questions():
    path = OUT / "traced.txt"
    if path.exists():
        return
    with open(path, "w", encoding="utf-8") as f:
        for question, folder in TRACED:
            doc_filter = path_filter(store, folder)
            if len(doc_filter) == 0:              # an empty filter would search the whole store
                print(folder, "is not in this store")
                continue
            context, line = retrieve(store, question, doc_filter)
            try:
                answer, seconds = ask_reader(question, context)
            except Exception as error:
                answer = "no answer: " + repr(error)[:200]
            f.write(f"{'=' * 100}\n{question}\nsearched {folder}, {line['tokens']} tokens\n\nANSWER: {answer}\n\nCONTEXT\n")
            f.write("\n\n".join(context) + "\n\n")
            print(question, "->", answer)


trace_questions()

## Block 18: the instruments

Numbers read from the logs and the store. They call no model.

- **The quote gate:** how many facts' quotes were found verbatim, and how many were refused.
- **Rejections:** by stage and reason, from Step 1's log.
- **Cost:** by stage and model, for the global layer, the reader and the judge.
- **Parents:** what merged parents join, with session copies counted apart.
- **Duplicate parents per history:** one name, inside one history, in two or more parents.
- **Parents in reach:** chat entities with a sibling in another session of their own history.
- **Summary agreement:** how many of a unit's entity narratives name an entity its unit summary also
  names.
- **The hand check:** 30 judged answers for a person to read.

```
quote gate, rejections   from Step 1's receipt and rejections.jsonl
cost                     sum calls.jsonl by stage and model
parents                  for each parent: its members' documents, sessions and histories
duplicates               for each history: names held by 2 or more parents
agreement                for each book unit: entity cells whose entity is named in the unit summary
hand check               HAND_CHECK judged answers of LongMemEval's best setting, drawn with the seed
```

In [ ]:
def cost_by_stage(path):
    """{stage and model: [calls, dollars]} from one call log."""
    totals = {}
    for row in read_jsonl(path):
        key = str(row.get("stage")) + ", " + str(row.get("model"))
        if key not in totals:
            totals[key] = [0, 0.0]
        totals[key][0] += 1
        totals[key][1] += row.get("cost") or 0.0
    for key in totals:
        totals[key][1] = round(totals[key][1], 2)
    return totals


def quote_gate():
    receipt = json.loads((STEP1_DIR / "receipt.json").read_text(encoding="utf-8"))
    matched = 0
    for how in receipt["matched_by"]:
        matched += receipt["matched_by"][how]
    refused = receipt["rejected_by"]["not_found"] + receipt["rejected_by"]["paraphrase"]
    return {"matched": matched, "matched_by": receipt["matched_by"], "rejected_by": receipt["rejected_by"],
            "pass_rate": share(matched, matched + refused), "cost_of_packages": receipt["cost_of_packages"]}


def rejections_by_stage():
    """Step 1's rejections by stage and reason: its table, or the rejection records of its packages."""
    paths = [STEP1_DIR / "rejections.jsonl"]
    if not paths[0].exists():
        paths = sorted(STEP1_DIR.rglob("*.jsonl"))
    counts = {}
    for path in paths:
        if path.name == "calls.jsonl":
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                if path.name != "rejections.jsonl" and row.get("record") != "rejection":
                    continue
                key = str(row.get("stage")) + ", " + str(row.get("category"))
                counts[key] = counts.get(key, 0) + 1
    return counts


def folded(name):
    return " ".join(name.casefold().split())


def parent_measures():
    """What parents join, and how far they reach inside one history."""
    db = store["db"]
    members = {}
    for doc_id, node_id, parent_id in db.execute("select doc_id, node_id, parent_id from instance_of"):
        if parent_id not in members:
            members[parent_id] = []
        members[parent_id].append((doc_id, node_id))

    joins = {"books only": 0, "books and chats": 0, "copies of one session": 0, "sessions of one history": 0,
             "sessions of several histories": 0}
    chat_entities = 0
    with_sibling = 0
    names_in_history = {}
    for parent_id in members:
        sessions = set()
        histories = {}
        books = 0
        for doc_id, node_id in members[parent_id]:
            uri = store["documents"][doc_id]["source_uri"]
            if "chats/" not in uri:
                books += 1
                continue
            path = PurePosixPath(uri)
            sessions.add(path.stem)
            if path.parent.name not in histories:
                histories[path.parent.name] = set()
            histories[path.parent.name].add(doc_id)
            name = folded(store["entity_names"][(doc_id, node_id)])
            key = (path.parent.name, name)
            if key not in names_in_history:
                names_in_history[key] = set()
            names_in_history[key].add(parent_id)

        for history in histories:
            for doc_id in histories[history]:
                count = 0
                for member in members[parent_id]:
                    if member[0] == doc_id:
                        count += 1
                chat_entities += count
                if len(histories[history]) > 1:
                    with_sibling += count

        if len(members[parent_id]) < 2:
            continue
        if books == len(members[parent_id]):
            joins["books only"] += 1
        elif books > 0:
            joins["books and chats"] += 1
        elif len(sessions) == 1:
            joins["copies of one session"] += 1
        elif len(histories) == 1:
            joins["sessions of one history"] += 1
        else:
            joins["sessions of several histories"] += 1

    duplicates = {}
    for history, name in names_in_history:
        if len(names_in_history[(history, name)]) > 1:
            duplicates[history] = duplicates.get(history, 0) + 1
    total = 0
    for history in duplicates:
        total += duplicates[history]
    return {"merged parents by what they join": joins, "chat entities": chat_entities,
            "with a sibling in another session of their history": with_sibling, "share": share(with_sibling, chat_entities),
            "duplicate names": total, "duplicate names per history": round(total / 500, 2)}


def summary_agreement():
    """Of the entity narratives in book units, how many name an entity the unit summary names."""
    db = store["db"]
    summaries = {}
    for doc_id, node_id, unit_id, text in db.execute("select doc_id, node_id, unit_id, text from cell"):
        if node_id == store["document_node"].get(doc_id):
            summaries[(doc_id, unit_id)] = text.casefold()
    named = 0
    total = 0
    for doc_id, node_id, unit_id in db.execute("select doc_id, node_id, unit_id from cell"):
        if node_id == store["document_node"].get(doc_id) or (doc_id, unit_id) not in summaries:
            continue
        total += 1
        name = store["entity_names"].get((doc_id, node_id), "").casefold()
        if name and name in summaries[(doc_id, unit_id)]:
            named += 1
    return {"unit summaries": len(summaries), "entity narratives": total, "named in the summary": named,
            "share": share(named, total)}


def write_hand_check():
    """HAND_CHECK judged answers of LongMemEval's best setting, drawn at random, to read by hand."""
    if "memory" not in BEST:
        return
    answers = read_rows(ANSWERS, "memory-" + BEST["memory"])
    rows = []
    for item in MEMORY:
        if item["id"] in answers:
            rows.append({"id": item["id"], "type": item["type"], "question": item["question"], "gold": item["answer"],
                         "response": answers[item["id"]]["response"], "correct": answers[item["id"]]["correct"]})
    random.Random(SEED).shuffle(rows)
    with open(OUT / "hand-check.jsonl", "w", encoding="utf-8") as f:
        for row in rows[:HAND_CHECK]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


INSTRUMENTS = {"quote gate": quote_gate(), "rejections": rejections_by_stage(),
               "global layer cost": cost_by_stage(STORE_DIR / "calls.jsonl"),
               "reader and judge cost": cost_by_stage(OUT / "calls.jsonl"),
               "parents": parent_measures(), "summary agreement": summary_agreement()}
for name in INSTRUMENTS:
    print(name, json.dumps(INSTRUMENTS[name], indent=1))
write_hand_check()

## Block 18b: the board's row

For each GraphRAG-Bench arm scored, its figures as the leaderboard shows them, from the evaluator's own
output: each question type's answer correctness averaged over the questions that type scored, the
Average as the plain mean of the four types, and ROUGE-L, coverage and faithfulness where the type has
them. The ids the evaluator dropped are listed. The arm's evaluator input and output are copied to
`submission/`, with the row beside them.

```
for each set and each of its arms with an evaluator output:
    for each question type: the mean of each metric over the rows that have it, and how many have it
    Average = mean of the four types' answer correctness
    dropped = the set's ids with no row in the output
    copy <set>-<arm>-input.json and <set>-<arm>-generation.json to submission/, and write <set>-<arm>-board.json
```

In [ ]:
BOARD_COLUMNS = {"Fact Retrieval": ("answer_correctness", "rouge_score"),
                 "Complex Reasoning": ("answer_correctness", "rouge_score"),
                 "Contextual Summarize": ("answer_correctness", "coverage_score"),
                 "Creative Generation": ("answer_correctness", "faithfulness", "coverage_score")}
SUBMISSION = OUT / "submission"


def board_row(set_name, arm_name):
    """The leaderboard's figures for one arm, from the evaluator's output file."""
    path = SCORES / (set_name + "-" + arm_name + "-generation.json")
    if not path.exists():
        return None
    results = json.loads(path.read_text(encoding="utf-8"))
    row = {"set": set_name, "arm": arm_name, "types": {}}
    in_output = set()
    accuracies = []
    for question_type in BOARD_COLUMNS:
        details = results.get(question_type, {}).get("detailed", [])
        for detail in details:
            in_output.add(detail["id"])
        cells = {}
        for metric in BOARD_COLUMNS[question_type]:
            values = []
            for detail in details:
                value = detail["metrics"].get(metric)
                if value is not None and not (isinstance(value, float) and math.isnan(value)):
                    values.append(value)
            cells[metric] = None
            if len(values) > 0:
                cells[metric] = round(100 * sum(values) / len(values), 2)
            if metric == "answer_correctness":
                cells["scored"] = len(values)
        row["types"][question_type] = cells
        accuracies.append(cells["answer_correctness"])
    row["average"] = None
    if None not in accuracies:
        row["average"] = round(sum(accuracies) / len(accuracies), 2)
    ids = set()
    for item in GRAPHRAG[set_name]:
        ids.add(item["id"])
    row["questions"] = len(ids)
    row["dropped"] = sorted(ids - in_output)
    return row


BOARD = []
for set_name in GRAPHRAG:
    for arm in arms_for(set_name):
        row = board_row(set_name, arm["name"])
        if row is None:
            continue
        BOARD.append(row)
        SUBMISSION.mkdir(exist_ok=True)
        name = set_name + "-" + arm["name"]
        for kind in ("input", "generation"):
            if (SCORES / (name + "-" + kind + ".json")).exists():
                shutil.copy(SCORES / (name + "-" + kind + ".json"), SUBMISSION / (name + "-" + kind + ".json"))
        (SUBMISSION / (name + "-board.json")).write_text(json.dumps(row, indent=2), encoding="utf-8")
        print(f"{name:32} Average {row['average']}   dropped {len(row['dropped'])} of {row['questions']}")
        for question_type in BOARD_COLUMNS:
            print(f"  {question_type:22}", row["types"][question_type])

## Block 19: results

Every table, in one file, with what the session finished. If `FINISHED` is false, attach this
output and run again.

In [ ]:
results = {"version": VERSION, "commit_of_scorer": SCORER_COMMIT, "finished": FINISHED, "memory": MEMORY_LINES, "graphrag_bench": GRAPHRAG_LINES,
           "approaches": APPROACHES, "board": BOARD,
           "published_memory": PUBLISHED_MEMORY, "published_plain_rag": PUBLISHED_PLAIN_RAG, "bodies": BODIES,
           "instruments": INSTRUMENTS, "spent": round(SPENT, 2)}
(OUT / "results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("finished:", FINISHED, f"; reader and judge ${SPENT:.2f}")

## References

1. **Reciprocal rank fusion (step 4).** G. V. Cormack, C. L. A. Clarke and S. Buettcher,
   "Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods", SIGIR 2009.
   The formula and k = 60 are theirs.
2. **BM25 (step 3).** S. Robertson and H. Zaragoza, "The Probabilistic Relevance Framework: BM25
   and Beyond", Foundations and Trends in Information Retrieval 3(4), 2009. Here it is SQLite
   FTS5's built-in `bm25()`.
3. **A record scored by its best sentence (step 1).** MaxP: Z. Dai and J. Callan, "Deeper Text
   Understanding for IR with Contextual Neural Language Modeling", SIGIR 2019, which scores a
   document by its best passage.
4. **The embedding (step 1).** `BAAI/bge-small-en-v1.5`: S. Xiao et al.,
   "C-Pack: Packaged Resources To Advance General Chinese Embedding", arXiv 2309.07597, 2023.
5. **Enter a graph by similarity and fill a token budget (steps 5 and 6).** Microsoft GraphRAG's
   local search (https://microsoft.github.io/graphrag/query/local_search/) enters through the
   entities most similar to the query, pulls their relationships and source text, and cuts the
   candidates to a fixed context window. LightRAG (Z. Guo et al., arXiv 2410.05779, 2024) and
   HippoRAG (B. Jimenez Gutierrez et al., NeurIPS 2024) expand from what they retrieve, by one hop
   and by Personalized PageRank. Here an approach either shows the hits (hops 0) or walks one or
   two hops from the best-placed of them (step 5b).
6. **Hybrid search over a temporal graph (steps 2 to 5 and 7).** Zep: P. Rasmussen, P. Paliychuk,
   T. Beauvais, J. Ryan and D. Chalef, "Zep: A Temporal Knowledge Graph Architecture for Agent
   Memory", arXiv 2501.13956, 2025. It searches by cosine similarity, BM25 and breadth-first search
   over n hops, reranks with RRF among others, and gives the model each fact with its date range.
7. **A parent standing for its instances (step 5).** The parent is the global layer's link between
   the same entity in different documents, so a parent hit is answered with those documents'
   abstracts, never with the parent itself.
8. **Search small, return large (step 6).** Sentence-window retrieval, as in LlamaIndex's
   `SentenceWindowNodeParser`: the vector is a sentence, the model reads the text around it. Here
   the fact is what is found and its passage of the source is what is read.